<a href="https://colab.research.google.com/github/farrelrassya/LLM-Design-Patterns-Building-Robust-and-Efficient-AI-Systems/blob/main/13.Quantization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 13 -- Quantization

*Companion notebook untuk* LLM Design Patterns

GPT-2 versi terkecil punya sekitar 124 juta parameter. Setiap parameter disimpan sebagai float32, empat byte per angka, sehingga seluruh model hampir 500 MB. *Quantization* menyimpan angka-angka itu dengan bit yang lebih sedikit, misalnya int8 yang hanya satu byte. Buku merangkum janjinya di Table 13.1: untuk PTQ (*post-training quantization*), penyimpanan berkurang 75% dan inferensi 2-4x lebih cepat.

Harganya adalah pembulatan. Int8 hanya punya 256 nilai, jadi setiap bobot dan setiap nilai yang mengalir di dalam model ikut dibulatkan. Bab ini menguji janji itu pada GPT-2 sungguhan dengan tiga pertanyaan: seberapa besar kerusakan akibat pembulatan, di bagian model mana kerusakan itu terjadi, dan apakah model yang lebih kecil memang lebih cepat.

Kode buku ditulis untuk model sederhana, sedangkan GPT-2 punya embedding, nilai ekstrem di aktivasinya, dan lapisan `Conv1D` alih-alih `nn.Linear`. Karena itu banyak listing buku gagal atau memberi hasil yang menyesatkan. Listing seperti itu ditampilkan sebagai kotak teks bertanda ⚠️, lengkap dengan error dan penyebabnya, dan sel kode sesudahnya menjalankan versi yang bekerja. Di empat titik ada pengukuran tambahan berjudul **Under the hood**. Jalankan sel berurutan dari atas, karena banyak sel memakai hasil sel sebelumnya.

## Intuisi: membulatkan ke 256 nilai

Sebelum memuat model apa pun, kita lihat inti quantization pada delapan angka. Int8 hanya punya 256 bilangan bulat, -128 sampai 127. Untuk menyimpan angka float sebagai int8, kita pilih sebuah *scale*: angka asli dibagi scale, dibulatkan ke bilangan bulat terdekat, dan bilangan bulat itulah yang disimpan, satu byte per angka. Saat dipakai, bilangan bulat itu dikali scale lagi. Di sel berikut scale dipilih supaya nilai absolut terbesar jatuh tepat di 127, dan angka 0 tetap 0 (*zero point* 0). Sel ini hanya butuh torch, yang sudah ada di Colab.

In [38]:
# Intuisi: simpan angka float sebagai int8, lalu baca kembali
import torch

def int8_roundtrip(x):
    scale = x.abs().max() / 127                                  # simetris: nilai terbesar -> 127, zero point 0
    q = torch.round(x / scale).clamp(-128, 127).to(torch.int8)   # yang disimpan: satu byte per angka
    return q, q.float() * scale, scale                           # yang dibaca kembali: q x scale

w = torch.tensor([-0.5, -0.31, -0.012, 0.0, 0.015, 0.1234, 0.27, 0.5])
w_outlier = torch.cat([w, torch.tensor([5.0])])
for label, x in (("delapan bobot", w), ("ditambah satu outlier 5.0", w_outlier)):
    q, x_hat, scale = int8_roundtrip(x)
    n = len(w)                                                   # tampilkan delapan bobot yang sama
    print(f"{label}: scale = {scale:.6f}")
    print(f"  int8 tersimpan : {q[:n].tolist()}")
    print(f"  dibaca kembali : {[round(v, 4) for v in x_hat[:n].tolist()]}")
    print(f"  galat terbesar : {(x - x_hat)[:n].abs().max():.4f} (setengah scale = {scale / 2:.4f})")

delapan bobot: scale = 0.003937
  int8 tersimpan : [-127, -79, -3, 0, 4, 31, 69, 127]
  dibaca kembali : [-0.5, -0.311, -0.0118, 0.0, 0.0157, 0.122, 0.2717, 0.5]
  galat terbesar : 0.0017 (setengah scale = 0.0020)
ditambah satu outlier 5.0: scale = 0.039370
  int8 tersimpan : [-13, -8, 0, 0, 0, 3, 7, 13]
  dibaca kembali : [-0.5118, -0.315, 0.0, 0.0, 0.0, 0.1181, 0.2756, 0.5118]
  galat terbesar : 0.0150 (setengah scale = 0.0197)


Tanpa outlier, scale-nya 0.5 / 127 = 0.003937. Bobot 0.1234 disimpan sebagai 31 dan dibaca kembali sebagai 0.122, dan galat terbesar (0.0017) tidak melebihi setengah scale.

Tambahkan satu nilai 5.0, dan scale harus naik sepuluh kali lipat supaya 5.0 terjangkau. Sekarang 0.1234 dibaca kembali sebagai 0.1181, dan dua bobot kecil, -0.012 dan 0.015, menjadi nol. **Rentang yang harus dijangkau satu scale menentukan seberapa kasar pembulatan untuk semua nilai lain.** Hampir semua yang terjadi di bab ini, termasuk mengapa listing buku merusak GPT-2, berawal dari prinsip ini.

## Fondasi

Sekarang kita siapkan runtime dan tiga hal yang dipakai sampai akhir bab: model yang akan dikompresi, penggaris untuk mengukur kerusakannya, dan alat untuk mengukur ukuran serta kecepatan. Notebook ini disiapkan untuk Google Colab dengan runtime CPU biasa; GPU tidak dipakai. Beberapa sel butuh beberapa menit.

> **Tentang output yang tersimpan.** Semua output berasal dari satu kali menjalankan notebook ini dari atas ke bawah, bukan di Colab, melainkan di Docker linux/amd64 lewat emulasi Rosetta di Mac (4 thread, torch 2.11 CPU, transformers 5.19). Saat dijalankan ulang, loss dan ukuran file biasanya dekat dengan angka di teks; angka model mini bisa bergeser, jadi cocokkan polanya. Milidetik dari emulasi tidak mewakili CPU fisik mana pun. Teks sesekali mengutip *run historis* di Xeon dengan AMX sebagai pembanding kecepatan.

Sel berikut memasang `torchao`, pustaka yang ditunjuk PyTorch sebagai pengganti API quantization lama, dalam versi yang cocok dengan torch di runtime ini, dan mematikan mode offline Hugging Face kalau ada. Kalau sel ini meminta *Runtime > Restart session*, lakukan, lalu jalankan ulang dari sel ini. Baris `Failed to load ... .so` di stderr hanya berarti kernel GPU torchao tidak dimuat.

In [39]:
# Persiapan runtime untuk Colab (CPU cukup); juga jalan di Jupyter lokal.
import importlib.metadata, importlib.util, os, subprocess, sys

for var in ("HF_HUB_OFFLINE", "HF_DATASETS_OFFLINE", "TRANSFORMERS_OFFLINE"):   # bab ini mengunduh dari Hub
    os.environ.pop(var, None)
hub_constants = sys.modules.get("huggingface_hub.constants")
if hub_constants is not None and hub_constants.HF_HUB_OFFLINE:
    raise RuntimeError("Mode offline Hugging Face sudah aktif di sesi ini: Runtime > Restart session, lalu jalankan ulang.")

def pip_install(*spec):
    env = dict(os.environ, PIP_ROOT_USER_ACTION="ignore", PIP_DISABLE_PIP_VERSION_CHECK="1")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *spec], env=env, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"pip install {' '.join(spec)} gagal:\n{(r.stderr or r.stdout)[-2000:]}")

for pkg in ("transformers", "datasets"):          # sudah ada di Colab; di luar Colab dipasang kalau belum ada
    if importlib.util.find_spec(pkg) is None:
        pip_install(pkg)

import torch

def torchao_pin(major_minor):
    # torchao 0.17/0.18 hanya cocok untuk torch >= 2.11; rilis lama mengikuti torch-nya masing-masing
    if major_minor >= (2, 15):
        return None                               # lebih baru dari tabel ini: rilis torchao terbaru
    if major_minor >= (2, 11):
        return "0.18.0"
    return {(2, 10): "0.16.0", (2, 9): "0.15.0"}.get(major_minor, "0.13.0")

pin = torchao_pin(tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2]))
try:
    installed = importlib.metadata.version("torchao")
except importlib.metadata.PackageNotFoundError:
    installed = None
if installed is None or (pin is not None and installed != pin):
    if "torchao" in sys.modules:                  # versi lain sudah ada di memori
        raise RuntimeError(f"torchao {installed} sudah ter-import: Runtime > Restart session, lalu jalankan ulang.")
    pip_install("--no-deps", "torchao" if pin is None else f"torchao=={pin}")   # --no-deps: torch tidak disentuh
    importlib.invalidate_caches()

import torchao
print(f"torch {torch.__version__} | torchao {torchao.__version__} "
      f"| GPU terdeteksi: {torch.cuda.is_available()} (bab ini tetap memakai CPU)")

torch 2.11.0+cpu | torchao 0.18.0 | GPU terdeteksi: False (bab ini tetap memakai CPU)


Token Hugging Face tidak wajib, karena GPT-2 dan wikitext-2 publik, tapi token bisa membantu kalau unduhan anonim dibatasi. Sel berikut memakai `HF_TOKEN` dari environment atau Colab Secrets kalau Hub menerimanya, tanpa pernah mencetaknya. Kalau tidak ada, ditolak Hub, atau izin aksesnya tidak diberikan, unduhan berjalan anonim dan tidak ada token yang dikirim. Token yang pernah ditolak baru dipakai lagi sesudah *Restart session*.

In [40]:
def use_hf_token():
    import huggingface_hub, huggingface_hub.constants
    token = os.environ.get("HF_TOKEN", "").strip()
    if not token:
        try:
            from google.colab import userdata    # hanya ada di Colab; bisa memunculkan dialog izin
            token = (userdata.get("HF_TOKEN") or "").strip()
        except Exception:                        # bukan Colab, secret tidak ada, akses ditolak, atau timeout
            token = ""
    if not token:
        reason = "tidak ada"
    elif os.environ.get("HF_HUB_DISABLE_IMPLICIT_TOKEN") == "1":
        return "pernah ditolak di sesi ini; Runtime > Restart session supaya token baru dipakai"
    else:
        try:
            huggingface_hub.whoami(token=token)  # cek token masih valid; info akunnya tidak dicetak
            os.environ["HF_TOKEN"] = token       # nilainya hanya disimpan di environment proses ini
            return "dipakai (nilainya tidak ditampilkan)"
        except Exception as e:
            reason = f"ditolak Hub ({type(e).__name__})"
    os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"   # jangan kirim token apa pun, termasuk dari Colab Secrets
    huggingface_hub.constants.HF_HUB_DISABLE_IMPLICIT_TOKEN = True
    return f"{reason}; unduhan anonim"

print("HF_TOKEN:", use_hf_token())

HF_TOKEN: pernah ditolak di sesi ini; Runtime > Restart session supaya token baru dipakai


Sekarang modelnya. Sel berikut memuat GPT-2 124M beserta tokenizer-nya dan mengambil teks wikitext-2: 8 potongan validasi masing-masing 128 token sebagai penggaris, dan potongan bagian *train* untuk kalibrasi dan training nanti. Sel ini juga meng-import `quant`, `quantize_dynamic`, dan `autocast` versi buku, karena listing buku yang memakainya hanya ditampilkan sebagai teks.

In [41]:
import os
import sys, io, time, math, copy, gc, collections, warnings, zlib
import torch, numpy as np
import torch.nn as nn
import torch.nn.functional as F
import transformers, datasets
from transformers import GPT2LMHeadModel, GPT2Tokenizer, GPT2Config, AutoTokenizer
from transformers.pytorch_utils import Conv1D
from datasets import load_dataset

# Nama-nama dari listing buku. Listing itu hanya tampil sebagai teks, jadi diimpor di sini.
import torch.quantization as quant
from torch.quantization import quantize_dynamic
try:
    from torch.cuda.amp import autocast           # autocast versi buku, dipakai di bagian mixed precision
except ImportError:                               # kalau suatu saat jalur lama ini dihapus dari torch
    import functools
    autocast = functools.partial(torch.amp.autocast, "cuda")

transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore", category=DeprecationWarning)   # deprecation torch.ao ditampilkan eksplisit nanti
warnings.filterwarnings("ignore", message=".*reduce_range will be deprecated.*")
warnings.filterwarnings("ignore", message=".*quantized tensor creation functions.*")
torch.manual_seed(42)
print("python:", sys.version.split()[0], "| torch:", torch.__version__, "| transformers:", transformers.__version__,
      "| datasets:", datasets.__version__, "| threads:", torch.get_num_threads())

gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")
gpt2_tok = GPT2Tokenizer.from_pretrained("gpt2")
gpt2_tok.pad_token = gpt2_tok.eos_token

wiki_val = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="validation")
wiki_train = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")

def pack(split, n_chars, seq):
    text = ""
    for t in split["text"]:
        if t.strip():
            text += t
        if len(text) >= n_chars:
            break
    ids = gpt2_tok(text, return_tensors="pt")["input_ids"][0]
    return ids[:len(ids) // seq * seq].view(-1, seq)

ft_val = pack(wiki_val, 16_000, 128)[:8]
ft_train = pack(wiki_train, 60_000, 128)

python: 3.13.16 | torch: 2.11.0+cpu | transformers: 5.18.0 | datasets: 4.8.5 | threads: 1


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Lalu alat ukurnya:

- `eval_gpt2`, penggaris kita: loss validasi rata-rata pada 8 potongan tadi.
- `nbytes`: ukuran checkpoint, yaitu jumlah byte `state_dict` saat disimpan ke disk.
- `latency_ms`: median waktu 7 forward pada batch 2 x 128 token, sesudah 2 forward pemanasan.

In [42]:
def eval_gpt2(m, data=ft_val):
    m.eval()
    with torch.no_grad():
        return float(np.mean([m(input_ids=data[i*2:(i+1)*2], labels=data[i*2:(i+1)*2]).loss.item()
                              for i in range(len(data)//2)]))

def nbytes(mm):
    torch.save(mm.state_dict(), "_size_probe.pt")
    n = os.path.getsize("_size_probe.pt")
    os.remove("_size_probe.pt")
    return n

def latency_ms(m, x=None, reps=7):
    x = ft_val[:2] if x is None else x
    m.eval()
    with torch.inference_mode():
        for _ in range(2):
            m(input_ids=x)
        ts = []
        for _ in range(reps):
            t0 = time.perf_counter()
            m(input_ids=x)
            ts.append(time.perf_counter() - t0)
    return 1000 * float(np.median(ts))

Dengan alat itu, kita ukur GPT-2 aslinya.

In [43]:
base_val = eval_gpt2(gpt2)
fp32_mb = nbytes(gpt2) / 1e6
census_lin = [n for n, mod in gpt2.named_modules() if isinstance(mod, nn.Linear)]
census_conv = sum(1 for _, mod in gpt2.named_modules() if isinstance(mod, Conv1D))
print(f"val loss baseline: {base_val:.3f} | checkpoint fp32: {fp32_mb:.1f} MB | "
      f"tied lm_head<->wte: {gpt2.lm_head.weight.data_ptr() == gpt2.transformer.wte.weight.data_ptr()}")
print(f"sensus GPT-2: nn.Linear = {len(census_lin)} ({census_lin[0]}) | Conv1D = {census_conv}")

val loss baseline: 3.762 | checkpoint fp32: 497.8 MB | tied lm_head<->wte: True
sensus GPT-2: nn.Linear = 1 (lm_head) | Conv1D = 48


Sel tadi mencetak dua baris: loss 3.762, checkpoint 497.8 MB, `lm_head` dan `wte` terikat (*tied*), serta sensus berisi satu `nn.Linear` dan 48 `Conv1D`.

Loss 3.762 adalah rata-rata *cross-entropy* tebakan token berikutnya, dalam satuan *nats* (logaritma natural). Cara membacanya: `e^3.762` kira-kira 43, jadi rata-rata model seyakin orang yang memilih di antara sekitar 43 kemungkinan yang sama besar. Tebakan asal di antara 50,257 token GPT-2 memberi sekitar 10.8. Sepanjang bab kita membaca *selisih* terhadap 3.762: +0.01 nats berarti peluang untuk token yang benar turun sekitar 1%, +0.1 sekitar 10%, dan +1 berarti peluang itu tinggal kira-kira sepertiganya.

Checkpoint 497.8 MB adalah 124.4 juta parameter kali 4 byte. Hampir sepertiganya embedding `wte` (50,257 x 768). GPT-2 memakai matriks itu dua kali, sebagai tabel embedding di awal dan sebagai `lm_head` di akhir, tapi menyimpannya sekali; itulah arti *tied*.

Sensusnya paling penting untuk bagian berikut: bagi PyTorch, satu-satunya `nn.Linear` di GPT-2 adalah `lm_head`. Ke-48 proyeksi di 12 blok semuanya `Conv1D`.

Satu pertanyaan sebelum listing pertama: apa kata torch sendiri tentang API quantization yang dipakai buku?

In [44]:
# Status API yang dipakai buku, ditanyakan langsung ke torch yang terpasang
import textwrap
from torch.ao.quantization import quantize_dynamic as _qd
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    _qd(nn.Sequential(nn.Linear(8, 8)), {nn.Linear}, dtype=torch.qint8)
print(f"torch {torch.__version__} | quantized engine: {torch.backends.quantized.engine} "
      f"| tersedia: {torch.backends.quantized.supported_engines}")
seen = set()
for w in caught:
    msg = " ".join(str(w.message).split())
    if msg in seen: continue
    seen.add(msg)
    print(f"\n[{w.category.__name__}]")
    print(textwrap.fill(msg, 110))


torch 2.11.0+cpu | quantized engine: x86 | tersedia: ['qnnpack', 'onednn', 'x86', 'fbgemm']

[DeprecationWarning]
torch.ao.quantization is deprecated and will be removed in 2.10. For migrations of users: 1. Eager mode
quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use
torchao eager mode quantize_ API instead 2. FX graph mode quantization
(torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to
use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 3. pt2e quantization has been migrated
to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) see
https://github.com/pytorch/ao/issues/2259 for more details


Peringatan itu menyatakan seluruh `torch.ao.quantization` sudah *deprecated* dan dijadwalkan dihapus di versi 2.10. Itu mencakup semua yang dipakai buku: `quantize_dynamic`, `prepare`, `convert`, dan `prepare_qat`. Untuk gaya *eager mode* yang dipakai buku, penggantinya `quantize_` di torchao. Run ini memakai torch 2.11, jadi tenggatnya sudah lewat, tapi API lama masih bisa dipanggil. Teks peringatannya bisa sedikit berbeda antarversi torch.

Baris pertama juga menyebut *quantized engine*, kumpulan kernel int8 di CPU. Untuk CPU Intel dan AMD bawaannya `x86`, gabungan kernel FBGEMM dan oneDNN. Listing buku masih memakai `'fbgemm'`, engine bawaan sebelumnya.

Rencana bab ini mengikuti pesan itu: listing buku memakai API lama, dan saat API lama tidak sanggup, torchao menjadi jalur perbaikannya.

## Understanding the basics

Quantization menyimpan dan menghitung angka dengan presisi lebih rendah: dari float32 ke float16 atau bfloat16 (2 byte), atau ke int8 (1 byte) seperti di sel intuisi tadi. Di setiap lapisan ada dua hal yang bisa di-quantize: bobot, yang tetap, dan *aktivasi*, yaitu input yang mengalir masuk dan berubah untuk setiap teks. Pada *dynamic quantization*, scale aktivasi dihitung ulang saat inferensi. Pada *static quantization*, scale itu dihitung sekali dari data contoh, lalu dibekukan.

Listing dynamic quantization pertama di buku menjanjikan *"This immediately reduces memory requirements and increases inference speed."*

> **⚠️ Listing buku (gagal): dynamic quantization**
>
> ```python
> model = ...
> quantized_model = quantize_dynamic(
>     model, {torch.nn.Linear}, dtype=torch.qint8
> )
> print(f"Original model size: {torch.cuda.memory_allocated()} bytes")
> print(f"Quantized model size: {torch.cuda.memory_allocated()} bytes")
> ```
>
> **Error:** `AttributeError: 'ellipsis' object has no attribute 'eval'`
>
> Di buku, `model = ...` berarti "isi dengan model Anda". Bagi Python, `...` adalah objek `Ellipsis`, dan `quantize_dynamic` gagal saat memanggil `model.eval()`. Placeholder seperti ini muncul lagi di beberapa listing. Sel berikut mengisinya dengan GPT-2, menjalankan sisa listing apa adanya, lalu mengukur ulang dengan alat ukur kita.

In [45]:
model = gpt2                                              # perbaikan: placeholder diisi LLM sungguhan
quantized_model = quantize_dynamic(
    model, {torch.nn.Linear}, dtype=torch.qint8
)
# Check size reduction -- persis buku
print(f"Original model size: {torch.cuda.memory_allocated()} bytes")
print(f"Quantized model size: {torch.cuda.memory_allocated()} bytes")

print("--- diukur ulang dengan penggaris yang benar ---")
qnames = [n for n, m in quantized_model.named_modules()
          if type(m).__module__.startswith("torch.ao.nn.quantized.dynamic")]
loss_book_dyn = eval_gpt2(quantized_model)
print(f"modul ter-quantisasi    : {len(qnames)} dari {len(census_lin) + census_conv} proyeksi -> {qnames}")
print(f"checkpoint              : {fp32_mb:.1f} MB -> {nbytes(quantized_model)/1e6:.1f} MB")
print(f"val loss                : {base_val:.3f} -> {loss_book_dyn:.3f}")
print(f"latensi forward bs2x128 : {latency_ms(model):.0f} ms -> {latency_ms(quantized_model):.0f} ms")
del quantized_model
gc.collect();                                             # ";" menahan angka hasil gc.collect() agar tidak tampil

Original model size: 0 bytes
Quantized model size: 0 bytes
--- diukur ulang dengan penggaris yang benar ---
modul ter-quantisasi    : 1 dari 49 proyeksi -> ['lm_head']
checkpoint              : 497.8 MB -> 536.4 MB
val loss                : 3.762 -> 8.995
latensi forward bs2x128 : 1277 ms -> 1134 ms


Dua baris ukuran dari buku mencetak `0`: `torch.cuda.memory_allocated()` membaca memori GPU, padahal model kita ada di CPU.

Alat ukur kita bercerita lain. Dari 49 proyeksi, hanya `lm_head` yang ter-quantisasi, persis seperti ramalan sensus. Checkpoint justru *membesar* dari 497.8 ke 536.4 MB: `lm_head` mendapat salinan int8 sendiri sebesar 38.6 MB, sementara `wte` tetap float32 sebagai tabel embedding, dan ikatan *tied* putus. Loss rusak dari 3.762 ke 8.995, mendekati tebakan asal, dan kecepatannya hanya sedikit berubah (1766 lawan 1556 ms). Itu bisa diduga: `lm_head` hanya memikul sekitar 31% perkalian di forward, jadi kalaupun ia gratis, percepatannya paling banter sekitar 1.45x.

Sasaran yang seharusnya adalah 48 `Conv1D` di dalam blok. Secara matematis `Conv1D` sama dengan lapisan linear, hanya bentuk bobotnya terbalik, (input, output). API quantization mencari `nn.Linear` dan tidak mengenali `Conv1D`, jadi sel berikut mendefinisikan `conv1d_to_linear`, yang mengganti setiap `Conv1D` dengan `nn.Linear` berbobot transpos. Fungsi ini menjadi kunci bab ini.

Sel sesudahnya memakai fungsi itu, lalu membandingkan enam skema: sasarannya (blok dan `lm_head`, blok saja, atau `lm_head` saja) dan cara menghitung scale bobot. *Per-tensor* berarti satu scale untuk seluruh matriks; *per-channel* berarti satu scale per baris. Ingat outlier 5.0 di sel intuisi: dengan per-channel, hanya baris yang memuatnya yang dibulatkan kasar.

In [46]:
def conv1d_to_linear(model):
    n = 0
    for _, mod in list(model.named_modules()):
        for cname, child in list(mod.named_children()):
            if isinstance(child, Conv1D):
                lin = nn.Linear(child.weight.shape[0], child.weight.shape[1])
                lin.weight = nn.Parameter(child.weight.detach().t().contiguous())   # Conv1D (in, out) -> Linear (out, in)
                lin.bias = nn.Parameter(child.bias.detach().clone())
                setattr(mod, cname, lin)
                n += 1
    return n

In [47]:
from torch.ao.quantization import default_dynamic_qconfig, per_channel_dynamic_qconfig
g = copy.deepcopy(gpt2)
n_conv = conv1d_to_linear(g)
print(f"conv1d_to_linear: {n_conv} Conv1D -> nn.Linear | val {eval_gpt2(g):.4f} (asli {base_val:.4f}) | "
      f"tied tetap: {g.lm_head.weight.data_ptr() == g.transformer.wte.weight.data_ptr()} | fp32 {latency_ms(g):.0f} ms\n")
schemes = [
    ("per-tensor, 48 blok + lm_head",       {nn.Linear: default_dynamic_qconfig}),
    ("per-channel, 48 blok + lm_head",      {nn.Linear: per_channel_dynamic_qconfig}),
    ("per-tensor, 48 blok (lm_head fp32)",  {"transformer.h": default_dynamic_qconfig}),
    ("per-channel, 48 blok (lm_head fp32)", {"transformer.h": per_channel_dynamic_qconfig}),
    ("per-tensor, lm_head saja",            {"lm_head": default_dynamic_qconfig}),
    ("per-channel, lm_head saja",           {"lm_head": per_channel_dynamic_qconfig}),
]
dyn_rows = {}
print(f"{'skema quantize_dynamic (qint8)':<38} {'Linear int8':>11} {'val loss':>9} {'MB':>7} {'ms':>5}")
for label, spec in schemes:
    q = quantize_dynamic(g, spec, dtype=torch.qint8)
    nq = sum(1 for m in q.modules() if type(m).__module__.startswith("torch.ao.nn.quantized.dynamic"))
    dyn_rows[label] = (eval_gpt2(q), nbytes(q) / 1e6, latency_ms(q))
    print(f"{label:<38} {nq:>11} {dyn_rows[label][0]:>9.3f} {dyn_rows[label][1]:>7.1f} {dyn_rows[label][2]:>5.0f}")
    del q
    gc.collect()

conv1d_to_linear: 48 Conv1D -> nn.Linear | val 3.7621 (asli 3.7621) | tied tetap: True | fp32 1246 ms

skema quantize_dynamic (qint8)         Linear int8  val loss      MB    ms
per-tensor, 48 blok + lm_head                   49     9.600   281.6   811
per-channel, 48 blok + lm_head                  49     9.204   283.8   815
per-tensor, 48 blok (lm_head fp32)              48     4.455   243.0   951
per-channel, 48 blok (lm_head fp32)             48     4.077   244.4   931
per-tensor, lm_head saja                         1     8.995   536.4  1539
per-channel, lm_head saja                        1     8.856   537.2  1156


`conv1d_to_linear` tidak mengubah perilaku model: loss tetap 3.7621 dan ikatan *tied* utuh. Salinan GPT-2 versi `nn.Linear` ini bernama `g` dan dipakai di hampir semua eksperimen berikutnya.

Di tabel, skema yang hanya menyasar 48 blok memangkas checkpoint ke sekitar 244 MB. Separuh, bukan seperempat, karena embedding `wte` (154 MB) masih float32. Per-channel jelas lebih baik untuk bobot: kerusakan turun dari +0.693 ke +0.315 nats dengan tambahan hanya 1.4 MB. Kolom ms kita bahas di Under the hood 4.

Yang aneh ada di `lm_head`. Setiap skema yang menyentuhnya berakhir di loss 8.9 sampai 9.6, dan per-channel hampir tidak menolong (8.995 ke 8.856). Kalau masalahnya bobot, per-channel seharusnya memperbaikinya. Petunjuknya: `quantize_dynamic` juga meng-quantize *input* setiap lapisan, dengan satu scale untuk seluruh tensor, dan di engine `x86` hanya dengan 7 bit (128 nilai) karena pengaturan `reduce_range`. Buku tidak pernah memisahkan kedua sumber kerusakan ini.

### Under the hood 1: bobot atau aktivasi?

Alatnya *fake quantization*: nilai di-quantize lalu langsung dikembalikan ke float. Nilai itu membawa galat pembulatan yang sama seperti versi int8-nya, tapi perhitungan tetap berjalan di float, sehingga kita bebas memilih apa yang dibulatkan: bobot saja, aktivasi saja, atau keduanya.

Sel berikut membuat simulator yang mengikuti rumus `torch.ao` dan mencobanya di `lm_head` saja dan di 48 blok saja. Ada satu pilihan baru untuk aktivasi, *per-token*: setiap token mendapat scale sendiri. Dua baris berlabel "resep torch" seharusnya mereproduksi angka `quantize_dynamic` di tabel sebelumnya, dan baris validasi di akhir output memeriksanya.

In [48]:
# === UNDER THE HOOD 1: bobot atau aktivasi -- siapa yang merusak int8 GPT-2? ===
gu = copy.deepcopy(g)
# lepas tie: seperti quantize_dynamic, lm_head memegang salinan sendiri
gu.lm_head.weight = nn.Parameter(gu.transformer.wte.weight.detach().clone())
W0 = {n: m.weight.detach().clone() for n, m in gu.named_modules() if isinstance(m, nn.Linear)}

def fq_w(W, per_channel, bits=8):
    # bobot: qint8 simetris, rumus observer default torch.ao
    qmax, qmin = 2 ** (bits - 1) - 1, -2 ** (bits - 1)
    amax = W.abs().amax(dim=1, keepdim=True) if per_channel else W.abs().max()
    scale = (amax / ((qmax - qmin) / 2)).clamp(min=1e-12)
    return (W / scale).round().clamp(qmin, qmax) * scale

def fq_a(x, mode, levels):
    # aktivasi: affine dinamis, per-tensor atau per-token
    mn, mx = ((x.min(), x.max()) if mode == "tensor"
              else (x.amin(-1, keepdim=True), x.amax(-1, keepdim=True)))
    mn, mx = mn.clamp(max=0.0), mx.clamp(min=0.0)
    scale = ((mx - mn) / levels).clamp(min=1e-12)
    zp = (-mn / scale).round()
    return ((x / scale + zp).round().clamp(0, levels) - zp) * scale

hooks = []
def set_fq(targets, w=None, a=None, levels=127):
    for h in hooks:
        h.remove()
    hooks.clear()
    for n, m in gu.named_modules():
        if isinstance(m, nn.Linear):
            sel = any(n.startswith(t) for t in targets)
            m.weight.data.copy_(fq_w(W0[n], w == "pc") if (sel and w) else W0[n])
            if sel and a:
                hooks.append(m.register_forward_pre_hook(
                    lambda mod, inp, a=a, lv=levels: (fq_a(inp[0], a, lv),)))

arms = [("W per-tensor", dict(w="pt")),
        ("W per-channel", dict(w="pc")),
        ("A per-tensor 7-bit", dict(a="tensor", levels=127)),
        ("A per-tensor 8-bit", dict(a="tensor", levels=255)),
        ("A per-token 7-bit", dict(a="token", levels=127)),
        ("W pt + A per-tensor 7-bit (= resep torch)", dict(w="pt", a="tensor", levels=127)),
        ("W pc + A per-tensor 7-bit (= resep torch)", dict(w="pc", a="tensor", levels=127)),
        ("W pc + A per-token 7-bit", dict(w="pc", a="token", levels=127))]
print(f"{'fake-quant (sisanya fp32)':<44} {'lm_head saja':>12} {'48 blok':>9}")
uth1 = {}
for label, kw in arms:
    r = []
    for tg in (["lm_head"], ["transformer.h"]):
        set_fq(tg, **kw)
        r.append(eval_gpt2(gu))
    uth1[label] = r
    print(f"{label:<44} {r[0]:>12.4f} {r[1]:>9.4f}")
set_fq([])
print(f"\nvalidasi simulator vs quantize_dynamic asli -- lm_head pt: "
      f"{uth1['W pt + A per-tensor 7-bit (= resep torch)'][0]:.4f} vs {dyn_rows['per-tensor, lm_head saja'][0]:.4f} | "
      f"48 blok pc: {uth1['W pc + A per-tensor 7-bit (= resep torch)'][1]:.4f} vs {dyn_rows['per-channel, 48 blok (lm_head fp32)'][0]:.4f}")

fake-quant (sisanya fp32)                    lm_head saja   48 blok
W per-tensor                                       3.7690    3.9592
W per-channel                                      3.7772    3.7603
A per-tensor 7-bit                                 8.8203    4.0460
A per-tensor 8-bit                                 5.1647    3.8453
A per-token 7-bit                                  5.7486    3.7707
W pt + A per-tensor 7-bit (= resep torch)          8.9952    4.4423
W pc + A per-tensor 7-bit (= resep torch)          8.8563    4.0550
W pc + A per-token 7-bit                           5.7759    3.7761

validasi simulator vs quantize_dynamic asli -- lm_head pt: 8.9952 vs 8.9952 | 48 blok pc: 4.0550 vs 4.0769


Simulatornya cukup setia: di `lm_head` hasilnya sama dengan `quantize_dynamic` sampai digit keempat (8.9952), dan di 48 blok selisihnya sekitar 0.02.

Hasilnya tegas: **kerusakan int8 di GPT-2 datang dari aktivasi, bukan dari bobot.** Bobot int8 sendirian hampir tidak merusak; di 48 blok, bobot per-channel bahkan tidak menambah loss (3.7603). Sebaliknya, aktivasi per-tensor 7-bit sendirian sudah membawa `lm_head` ke 8.82, hampir seluruh kerusakan tadi. Satu bit tambahan hanya memperbaikinya ke 5.16.

Aktivasi per-token hampir gratis untuk 48 blok (3.771, +0.009), tapi `lm_head` tetap rusak di 5.75. Jadi ada sesuatu yang khusus pada input `lm_head`. Sel berikut mengintip input beberapa lapisan pada dua potongan teks: nilai absolut terbesar, mediannya, rasio keduanya, dan tiga dimensi dengan nilai terbesar. Nilai yang jauh lebih besar daripada nilai lain seperti ini disebut *outlier*.

In [49]:
stats = {}
def cap(name):
    def f(mod, inp):
        x = inp[0].detach().abs().flatten(0, -2)                     # (token, fitur)
        stats[name] = (x.max().item(), x.median().item(),
                       x.amax(0).topk(3).indices.tolist(), x.amax(0).topk(3).values.tolist())
    return f
hs = [m.register_forward_pre_hook(cap(n)) for n, m in gu.named_modules() if isinstance(m, nn.Linear)]
with torch.no_grad():
    gu(input_ids=ft_val[:2])
for h in hs:
    h.remove()
print(f"\n{'input ke layer':<29} {'max|x|':>8} {'median|x|':>10} {'rasio':>6}   3 dimensi outlier teratas")
for n in ["transformer.h.0.attn.c_attn", "transformer.h.0.mlp.c_fc", "transformer.h.0.mlp.c_proj",
          "transformer.h.5.attn.c_attn", "transformer.h.11.mlp.c_fc", "transformer.h.11.mlp.c_proj", "lm_head"]:
    mx, med, idx, vals = stats[n]
    print(f"{n:<29} {mx:>8.2f} {med:>10.3f} {mx/med:>6.0f}   {idx} = {[round(v, 1) for v in vals]}")
del gu, W0
gc.collect();                                 # ";" menahan angka hasil gc.collect() agar tidak tampil


input ke layer                  max|x|  median|x|  rasio   3 dimensi outlier teratas
transformer.h.0.attn.c_attn       0.80      0.073     11   [674, 709, 480] = [0.8, 0.8, 0.7]
transformer.h.0.mlp.c_fc          2.85      0.083     34   [674, 640, 55] = [2.9, 1.8, 1.6]
transformer.h.0.mlp.c_proj       10.88      0.117     93   [1612, 2754, 1567] = [10.9, 5.3, 5.3]
transformer.h.5.attn.c_attn       7.87      0.163     48   [64, 480, 266] = [7.9, 6.5, 5.1]
transformer.h.11.mlp.c_fc         3.52      0.125     28   [640, 393, 77] = [3.5, 2.5, 2.3]
transformer.h.11.mlp.c_proj      24.26      0.132    183   [2378, 3066, 611] = [24.3, 8.5, 8.1]
lm_head                         330.19      0.309   1069   [496, 430, 36] = [330.2, 231.8, 202.0]


Input `lm_head` adalah keluaran `ln_f`, LayerNorm terakhir GPT-2. Di sana dimensi 496 mencapai 330, padahal median nilai absolutnya hanya 0.31: rasionya 1069. Ini cerita bobot 5.0 dalam versi ekstrem. Satu scale dengan 128 nilai harus menjangkau 330, jadi jarak antarnilainya sedikitnya 2.6, dan setiap nilai di bawah sekitar 1.3 menjadi nol. Sedikitnya separuh input `lm_head` hilang. Per-token pun tidak menolong, kemungkinan karena dimensi-dimensi itu besar di sebagian besar token, sehingga hampir setiap token tetap mendapat scale yang kasar.

Outlier juga ada di dalam blok, walaupun jauh lebih kecil. Di antara input blok yang kita intip, rasio tertinggi ada di `mlp.c_proj`, sesudah GELU: 93 di layer 0 dan 183 di layer 11. Angka ini dipakai lagi di bagian mixed precision.

Diagnosisnya lengkap. Untuk 48 blok: bobot per-channel dan aktivasi per-token. Untuk `lm_head`: jangan quantize aktivasinya, cukup bobotnya (*weight-only*). `torch.ao` tidak menyediakan aktivasi per-token, tapi torchao menyediakannya.

In [50]:
# Jalur migrasi resmi (pesan deprecation di atas): torchao quantize_
# -- aktivasi int8 PER-TOKEN + bobot int8 per-channel
import torchao   # dipasang oleh sel persiapan; bukan resep buku, tapi jalur yang ditunjuk torch sendiri
from torchao.quantization import quantize_, Int8DynamicActivationInt8WeightConfig, Int8WeightOnlyConfig

def ao_int8(model, embeddings=True):
    if embeddings:
        # embedding: weight-only (aktivasi tidak di-quantize -- outlier ln_f di input lm_head tak disentuh)
        quantize_(model, Int8WeightOnlyConfig(), filter_fn=lambda m, fqn: fqn == "transformer.wte")
        model.lm_head.weight = model.transformer.wte.weight            # quantize_ memutus tie; pasang lagi
    quantize_(model, Int8DynamicActivationInt8WeightConfig(),
              filter_fn=lambda m, fqn: isinstance(m, nn.Linear) and fqn.startswith("transformer.h"))
    return model

print(f"torchao {torchao.__version__}")
ao_rows = {}
for label, emb in (("torchao W8A8 per-token (48 blok)", False), ("torchao W8A8 + embedding W8 (tied)", True)):
    q = ao_int8(copy.deepcopy(g), embeddings=emb)
    ao_rows[label] = (eval_gpt2(q), nbytes(q) / 1e6, latency_ms(q))
    v, mb, ms = ao_rows[label]
    print(f"{label:<36}: val {v:.3f} ({v - base_val:+.3f}) | {mb:.1f} MB ({mb / fp32_mb - 1:+.0%}) | {ms:.0f} ms "
          f"| tied: {q.lm_head.weight is q.transformer.wte.weight}")
    del q
    gc.collect()

torchao 0.18.0
torchao W8A8 per-token (48 blok)    : val 3.764 (+0.002) | 243.5 MB (-51%) | 20960 ms | tied: True
torchao W8A8 + embedding W8 (tied)  : val 3.780 (+0.018) | 127.9 MB (-74%) | 21652 ms | tied: True


`Int8DynamicActivationInt8WeightConfig` mengerjakan resep tadi: bobot int8 per-channel dan aktivasi int8 per-token. Diterapkan ke 48 blok, loss-nya 3.764, hanya +0.002, dengan checkpoint 243.5 MB. Ukurannya sama dengan `torch.ao` per-channel, tapi kerusakannya turun dari +0.315 menjadi praktis nol.

Baris kedua menambahkan embedding: bobot `wte` di-quantize dengan `Int8WeightOnlyConfig`, lalu `lm_head` diarahkan kembali ke `wte` karena `quantize_` memutus ikatan *tied*. Hasilnya 127.9 MB, **74% lebih kecil dengan loss hanya +0.018 nats**. Kuncinya dua hal yang tidak ada di listing buku: tahu di mana outlier berada, dan tahu lapisan mana yang aktivasinya tidak boleh di-quantize.

Soal kecepatan, varian dengan embedding int8 tidak lebih cepat daripada varian blok saja (2936 lawan 2853 ms), kemungkinan karena embedding harus di-*dequantize* setiap kali dipakai. Byte dan milidetik adalah dua tujuan yang berbeda; Under the hood 4 menimbang keduanya. `ao_int8` dipakai lagi sampai akhir bab, jadi kalau sel ini error, periksa dulu versi torchao di output sel persiapan.

### Static quantization

Pada static quantization, scale aktivasi dihitung sekali dari data contoh sebelum model dipakai, lalu dibekukan. Proses ini disebut *kalibrasi*. Yang merekam rentang nilai selama kalibrasi adalah *observer*, modul kecil yang ditempelkan pada keluaran lapisan. Observer mana yang dipakai ditentukan *qconfig*, resep berisi observer untuk aktivasi dan untuk bobot. Qconfig dipasang lewat atribut `.qconfig` dan diwariskan ke semua submodul; detail ini nanti menjadi masalah di bagian PTQ.

Buku mendemonstrasikannya pada `SimpleModel` kecil, dua lapisan linear (784 ke 256 ke 10) dengan ReLU, dalam tiga langkah: `prepare` menyisipkan observer, 100 input `torch.randn` dilewatkan sebagai kalibrasi, lalu `convert` mengganti lapisannya dengan versi int8.

> **⚠️ Listing buku (gagal): static quantization pada `SimpleModel`**
>
> ```python
> # ... definisi SimpleModel, prepare, dan kalibrasi seperti di buku ...
> quantized_model = torch.quantization.convert(prepared_model)
> # Model is now statically quantized and ready for inference
> print(quantized_model
> ```
>
> **Error:** `SyntaxError: '(' was never closed`
>
> Python menolak sel ini saat membacanya, sebelum satu baris pun berjalan, karena baris terakhir kehilangan kurung tutup. Sel berikut menjalankan listing yang sama, dengan `)` di tempatnya.

In [51]:
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(784, 256)
        self.relu = nn.ReLU()
        self.out = nn.Linear(256, 10)
    def forward(self, x):
        x = self.relu(self.fc(x))
        return self.out(x)
model_fp32 = SimpleModel()
model_fp32.eval()
model_fp32.qconfig = torch.quantization.get_default_qconfig(
    'fbgemm')
prepared_model = torch.quantization.prepare(model_fp32)
for _ in range(100):
    sample_input = torch.randn(1, 784)
    prepared_model(sample_input)
quantized_model = torch.quantization.convert(prepared_model)
print(quantized_model)                                    # perbaikan: kurung ditutup

SimpleModel(
  (fc): QuantizedLinear(in_features=784, out_features=256, scale=0.03519655764102936, zero_point=65, qscheme=torch.per_channel_affine)
  (relu): ReLU()
  (out): QuantizedLinear(in_features=256, out_features=10, scale=0.011033361777663231, zero_point=63, qscheme=torch.per_channel_affine)
)


Kedua `nn.Linear` kini `QuantizedLinear` dengan bobot `per_channel_affine`, satu scale per baris. Pasangan `scale` dan `zero_point` yang tercetak adalah parameter untuk output setiap lapisan, dibekukan dari 100 sampel kalibrasi. Listing tidak memasang seed, jadi angka ini berubah setiap kali sel dijalankan ulang.

Komentar listing menyebut model ini *"ready for inference"*, tapi listing-nya tidak pernah memanggil model itu.

> **⚠️ Model hasil listing dipanggil (gagal)**
>
> ```python
> quantized_model(torch.randn(1, 784))
> ```
>
> **Error:** `NotImplementedError: Could not run 'quantized::linear' with arguments from the 'CPU' backend. ...`
>
> `quantized::linear` hanya punya kernel untuk tensor yang *sudah* ter-quantisasi, sedangkan yang kita berikan tensor float biasa. Dalam static quantization gaya *eager*, batas float dan int8 harus ditandai sendiri: `QuantStub` di pintu masuk mengubah input float menjadi `quint8`, dan `DeQuantStub` di pintu keluar mengembalikannya ke float. `SimpleModel` buku tidak punya keduanya.

Perbaikannya empat baris: dua stub di `__init__` dan dua panggilan di `forward`. Kalibrasinya tetap seperti buku, kali ini dengan seed tetap. Sesudah konversi, output model int8 dibandingkan dengan float32 pada 256 input uji.

In [52]:
class SimpleModelQ(nn.Module):                             # perbaikan: pintu masuk/keluar float <-> int8
    def __init__(self):
        super().__init__()
        self.quant = torch.quantization.QuantStub()
        self.fc = nn.Linear(784, 256)
        self.relu = nn.ReLU()
        self.out = nn.Linear(256, 10)
        self.dequant = torch.quantization.DeQuantStub()
    def forward(self, x):
        x = self.quant(x)
        x = self.relu(self.fc(x))
        return self.dequant(self.out(x))

torch.manual_seed(0)
m_fp32 = SimpleModelQ().eval()
m_fp32.qconfig = torch.quantization.get_default_qconfig('fbgemm')
m_prep = torch.quantization.prepare(copy.deepcopy(m_fp32))
for _ in range(100):                                       # kalibrasi persis buku: torch.randn
    m_prep(torch.randn(1, 784))
m_int8 = torch.quantization.convert(m_prep)
print(m_int8)
x_test = torch.randn(256, 784)
with torch.no_grad():
    y32, y8 = m_fp32(x_test), m_int8(x_test)
print(f"\n256 input uji: max |y_fp32 - y_int8| = {(y32 - y8).abs().max():.4f} (rata-rata |y_fp32| = {y32.abs().mean():.4f}) "
      f"| argmax sama: {(y32.argmax(1) == y8.argmax(1)).float().mean():.1%}")
print(f"state_dict: {nbytes(m_fp32)/1e3:.0f} KB -> {nbytes(m_int8)/1e3:.0f} KB")

SimpleModelQ(
  (quant): Quantize(scale=tensor([0.0634]), zero_point=tensor([64]), dtype=torch.quint8)
  (fc): QuantizedLinear(in_features=784, out_features=256, scale=0.03787487745285034, zero_point=59, qscheme=torch.per_channel_affine)
  (relu): ReLU()
  (out): QuantizedLinear(in_features=256, out_features=10, scale=0.011293431743979454, zero_point=66, qscheme=torch.per_channel_affine)
  (dequant): DeQuantize()
)

256 input uji: max |y_fp32 - y_int8| = 0.0530 (rata-rata |y_fp32| = 0.1851) | argmax sama: 97.3%
state_dict: 817 KB -> 214 KB


Sekarang model int8-nya bekerja. `Quantize` di pintu masuk punya scale 0.0634 dan zero point 64, cukup untuk menjangkau kira-kira -4 sampai 4 dengan 128 nilai, pas untuk input `torch.randn`. Zero point 64 berada di tengah 0 sampai 127 karena inputnya tersebar di sekitar nol. Hanya 128 nilai, bukan 256 dari `quint8`, karena qconfig ini juga memakai `reduce_range`.

Pada 256 input uji, 97.3% prediksi sama dengan model float32, dan `state_dict` menyusut dari 817 ke 214 KB, turun 74%. Di model sekecil ini klaim penghematan sekitar 75% terpenuhi dengan mudah: tidak ada embedding, dan data uji berasal dari distribusi yang sama dengan data kalibrasi. Di GPT-2 kedua kemudahan itu tidak ada.

### PTQ

*Post-training quantization* (PTQ) berarti meng-quantize model yang sudah selesai dilatih. Dynamic dan static quantization tadi sama-sama PTQ. Untuk yang statis, prosa buku menegaskan bahwa PTQ *"often require a calibration step on a representative dataset"*. Listing PTQ buku memakai urutan yang sama dengan `SimpleModel`, yaitu pasang qconfig, `prepare`, lalu `convert`, kali ini untuk LLM pretrained.

> **⚠️ Listing buku (gagal): PTQ**
>
> ```python
> model = ...
> model.eval()
> model.qconfig = torch.quantization.default_qconfig
> model_prepared = quant.prepare(model)
> model_quantized = quant.convert(model_prepared)
> ```
>
> **Error:** `AttributeError: 'ellipsis' object has no attribute 'eval'`, placeholder yang sama seperti listing pertama.
>
> Kalau placeholder diisi salinan GPT-2, `convert` berhenti di error berikutnya: `AssertionError: Embedding quantization is only supported with float_qparams_weight_only_qconfig.` Qconfig yang dipasang di akar model diwariskan ke embedding `wte` dan `wpe`, padahal embedding butuh skema yang hanya meng-quantize bobot. Seandainya langkah itu lolos pun, GPT-2 tidak punya `QuantStub`, dan 48 `Conv1D`-nya tetap tak terlihat.

Perbaikan yang tetap setia pada maksud listing: pakai `g`, bungkus setiap proyeksi blok dengan `QWrap` (berisi `QuantStub`, lapisan aslinya, dan `DeQuantStub`), lalu pasang qconfig buku *hanya* di 48 pembungkus itu. Embedding, LayerNorm, dan GELU tetap float. Sesudahnya urutan buku dijalankan apa adanya: `prepare`, lalu langsung `convert`, tanpa kalibrasi, karena listing-nya memang tidak memuat langkah itu.

In [53]:
class QWrap(nn.Module):
    # perbaikan: stub di sekeliling tiap Linear; embedding/LN/GELU tetap float
    def __init__(self, lin):
        super().__init__()
        self.quant = torch.quantization.QuantStub()
        self.lin = lin
        self.dequant = torch.quantization.DeQuantStub()
    def forward(self, x):
        return self.dequant(self.lin(self.quant(x)))

def wrap_blocks(base, qconfig):
    m = copy.deepcopy(base)
    for blk in m.transformer.h:
        for parent, names in ((blk.attn, ("c_attn", "c_proj")), (blk.mlp, ("c_fc", "c_proj"))):
            for nm in names:
                w = QWrap(getattr(parent, nm))
                w.qconfig = qconfig
                setattr(parent, nm, w)
    return m.eval()

model = wrap_blocks(g, torch.quantization.default_qconfig)   # qconfig persis buku, hanya di 48 pembungkus
model_prepared = quant.prepare(model)
model_quantized = quant.convert(model_prepared)              # urutan persis buku: prepare -> convert, tanpa kalibrasi
st = model_quantized.transformer.h[0].attn.c_attn.quant
print(f"skala input h0.attn.c_attn: scale = {st.scale.item()}, zero_point = {st.zero_point.item()}")
print(f"val loss: {base_val:.3f} -> {eval_gpt2(model_quantized):.3f}")

/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/observer.py:368: UserWarning: must run observer before calling calculate_qparams. Returning default values.
  if not check_min_max_valid(min_val, max_val):


skala input h0.attn.c_attn: scale = 1.0, zero_point = 0
val loss: 3.762 -> 8.079


Model hasil konversi bisa dipanggil, tapi hampir tidak berguna: loss naik dari 3.762 ke 8.079. Penyebabnya terlihat di output. Scale input `h0.attn.c_attn` bernilai 1.0 dengan zero point 0, dan torch sendiri memperingatkan *"must run observer before calling calculate_qparams"*. Observer yang disisipkan `prepare` tidak pernah melihat satu aktivasi pun, jadi semua scale jatuh ke nilai bawaan.

Lihat akibatnya pada input `c_attn` di layer 0, keluaran `ln_1` yang median nilai absolutnya hanya sekitar 0.073. Dengan jarak antarnilai 1.0, hampir semua nilai dibulatkan ke nol. Zero point 0 juga tidak menyisakan tempat untuk angka negatif di `quint8`, jadi nilai negatif ikut terpotong. Prosa buku menyebut kalibrasi sering dibutuhkan; listing-nya justru tidak memuat langkah itu.

### Under the hood 2: kalibrasi

Seberapa penting kalibrasi, dan apakah datanya harus teks sungguhan? Sel berikut membandingkan tiga pilihan: tanpa kalibrasi (urutan listing buku), token acak (padanan `torch.randn` untuk model bahasa: 8 x 128 nomor token acak), dan teks asli dari bagian *train* wikitext-2. Masing-masing dicoba dengan dua observer aktivasi. *MinMax* memakai nilai terkecil dan terbesar yang pernah lewat, jadi wajib menjangkau outlier terjauh. *Histogram*, bawaan `'fbgemm'` dan `'x86'`, memilih rentang yang meminimalkan galat pembulatan walaupun harus memotong segelintir nilai ekstrem. Bobot di semua varian memakai per-channel.

In [54]:
# === UNDER THE HOOD 2: kalibrasi PTQ statis -- tanpa, token acak, teks riil; MinMax vs Histogram ===
from torch.ao.quantization import QConfig, MinMaxObserver, HistogramObserver, PerChannelMinMaxObserver
del model, model_prepared, model_quantized                 # tiga salinan GPT-2 dari sel sebelumnya dilepas
gc.collect()
calib_real = ft_train[:8]                                  # 8 x 128 token teks wikitext-train (bukan penggaris val)
# padanan torch.randn buku untuk model bahasa: nomor token acak
calib_rand = torch.randint(0, 50257, (8, 128), generator=torch.Generator().manual_seed(0))
w_obs = PerChannelMinMaxObserver.with_args(dtype=torch.qint8, qscheme=torch.per_channel_symmetric)
observers = {"MinMax": QConfig(activation=MinMaxObserver.with_args(reduce_range=True), weight=w_obs),
             "Histogram": QConfig(activation=HistogramObserver.with_args(reduce_range=True), weight=w_obs)}

def static_ptq(qconfig, calib):
    p = quant.prepare(wrap_blocks(g, qconfig))
    if calib is not None:
        with torch.no_grad():
            for i in range(0, len(calib), 2):
                p(input_ids=calib[i:i + 2])
    return quant.convert(p)

print(f"{'kalibrasi':<22} {'observer':<10} {'skala h0.c_attn':>15} {'val loss':>9} {'delta':>7}")
uth2 = {}
for cname, calib in (("tanpa (urutan buku)", None), ("token acak", calib_rand), ("teks riil", calib_real)):
    for oname, qc in observers.items():
        q = static_ptq(qc, calib)
        v = eval_gpt2(q)
        sc = q.transformer.h[0].attn.c_attn.quant.scale.item()
        uth2[(cname, oname)] = v
        print(f"{cname:<22} {oname:<10} {sc:>15.5f} {v:>9.3f} {v - base_val:>+7.3f}")
        del q
        gc.collect()
print(f"\nreferensi -- dynamic per-channel torch.ao (48 blok): {dyn_rows['per-channel, 48 blok (lm_head fp32)'][0]:.3f} "
      f"| torchao W8A8 per-token (48 blok): {ao_rows['torchao W8A8 per-token (48 blok)'][0]:.3f}")

kalibrasi              observer   skala h0.c_attn  val loss   delta
tanpa (urutan buku)    MinMax             1.00000     8.080  +4.318


/usr/local/lib/python3.13/dist-packages/torch/ao/nn/quantized/modules/__init__.py:133: UserWarning: must run observer before calling calculate_qparams.                                    Returning default scale and zero point 
  scale, zero_point = mod.activation_post_process.calculate_qparams()
/usr/local/lib/python3.13/dist-packages/torch/ao/nn/quantized/modules/linear.py:334: UserWarning: must run observer before calling calculate_qparams.                                    Returning default scale and zero point 
  act_scale, act_zp = activation_post_process.calculate_qparams()


tanpa (urutan buku)    Histogram          1.00000     8.080  +4.318
token acak             MinMax             0.01247     5.373  +1.611
token acak             Histogram          0.00880     4.844  +1.082
teks riil              MinMax             0.01245     5.384  +1.622
teks riil              Histogram          0.01065     4.775  +1.013

referensi -- dynamic per-channel torch.ao (48 blok): 4.077 | torchao W8A8 per-token (48 blok): 3.764


Tanpa kalibrasi, kedua observer menyerah ke scale 1.0 dan loss tertahan di 8.080. Itulah model yang dihasilkan listing PTQ buku. (Selisih 0.001 dengan 8.079 tadi datang dari bobot: di sini per-channel, di listing per-tensor.)

Begitu kalibrasi dijalankan, yang paling menentukan adalah jenis observer. Histogram unggul sekitar 0.5-0.6 nats atas MinMax, karena rela memotong nilai ekstrem sehingga jarak antarnilainya lebih rapat; lagi-lagi prinsip bobot 5.0. Jenis data hampir tidak berpengaruh: **token acak hampir sama baiknya dengan teks asli** (4.844 lawan 4.775 dengan Histogram). Angka ini paling mudah dijelaskan kalau outlier GPT-2 bersifat struktural, muncul di dimensi yang sama apa pun teksnya.

Meski begitu, PTQ statis terbaik (+1.013) masih jauh di belakang PTQ dinamis: +0.315 untuk `torch.ao` per-channel dan +0.002 untuk torchao per-token. Kemungkinan penyebabnya dua, walaupun tidak dipisahkan di sini: scale aktivasi yang beku untuk semua token, dan pembulatan tambahan pada output setiap lapisan. Buku menyebut PTQ *"generally maintains accuracy within acceptable limits"*; untuk PTQ statis dengan API buku di GPT-2, varian terbaik masih kehilangan sekitar 1 nats.

### QAT

*Quantization-aware training* (QAT) mensimulasikan pembulatan selama training, sehingga model bisa belajar menyesuaikan diri. Di setiap forward, bobot dan aktivasi di-fake-quantize seperti di Under the hood 1. Masalahnya, pembulatan adalah fungsi bertangga yang turunannya nol hampir di mana-mana. QAT memakai *straight-through estimator* (STE): saat backward, pembulatan dianggap tidak ada, dan gradien diteruskan ke bobot float di belakangnya. Bobot float itu baru di-quantize sungguhan setelah training selesai.

Buku menjanjikan akurasi lebih baik daripada PTQ, *"particularly for more complex tasks"*, tapi listing-nya hanya kerangka.

> **⚠️ Listing buku (gagal): kerangka QAT**
>
> ```python
> import torch.quantization as quant
> # Set up QAT
> model.train()
> model.qconfig = torch.quantization.get_default_qat_qconfig('fbgemm')
> model_prepared = quant.prepare_qat(model)
> for epoch in range(num_epochs):
>     train_one_epoch(model_prepared, train_loader, optimizer)
>     validate(model_prepared, val_loader)
> model_quantized = quant.convert(model_prepared.eval())
> ```
>
> **Error** (di kernel baru): `NameError: name 'model' is not defined`
>
> Kerangka ini tidak pernah mendefinisikan `model`, `num_epochs`, `train_one_epoch`, `train_loader`, `optimizer`, `validate`, maupun `val_loader`, dan tidak meng-import `torch`. Di notebook biasa sebagian kesalahan ini tertutup oleh sel-sel sebelumnya.

Untuk mengisi kerangka itu kita butuh model yang bisa dilatih dalam hitungan menit di CPU. Jadi kita pakai GPT-2 mini: arsitekturnya sama, tapi hanya 2 layer dengan vektor 64 dimensi, dilatih dari nol. Korpusnya 20,000 baris pertama wikitext-2 bagian train, dengan kosakata dipangkas ke 4,095 token tersering ditambah `<unk>`. Model mini dilatih 2,000 langkah dalam float32, dan checkpoint itulah titik awal semua percobaan QAT.

In [55]:
# Korpus proxy + GPT-2 mini untuk percobaan QAT
SEQ_LEN, VOCAB = 64, 4096
tok_mini = AutoTokenizer.from_pretrained("gpt2")
texts = [t for t in wiki_train["text"][:20000] if t.strip()]
raw_ids = []
for out in tok_mini(texts)["input_ids"]:
    raw_ids.extend(out + [tok_mini.eos_token_id])
freq = collections.Counter(raw_ids)
remap = {t: i + 1 for i, (t, _) in enumerate(freq.most_common(VOCAB - 1))}   # id 0 = <unk>
ids = torch.tensor([remap.get(t, 0) for t in raw_ids])
blocks = ids[:len(ids) // SEQ_LEN * SEQ_LEN].view(-1, SEQ_LEN)
perm = torch.randperm(len(blocks), generator=torch.Generator().manual_seed(0))
val_blocks, tr_blocks = blocks[perm[:256]], blocks[perm[256:]]
print(f"korpus proxy: {len(raw_ids):,} token | cakupan vocab {VOCAB}: {(ids > 0).float().mean():.1%} "
      f"| blok train {len(tr_blocks)} / val {len(val_blocks)}")

def make_mini(dropout=0.0):
    return GPT2LMHeadModel(GPT2Config(
        vocab_size=VOCAB, n_positions=SEQ_LEN, n_embd=64, n_layer=2, n_head=2,
        resid_pdrop=dropout, embd_pdrop=dropout, attn_pdrop=dropout, bos_token_id=0, eos_token_id=0))

def eval_mini(m):
    m.eval()
    with torch.no_grad():
        return float(np.mean([m(input_ids=val_blocks[i:i + 32], labels=val_blocks[i:i + 32]).loss.item()
                              for i in range(0, len(val_blocks), 32)]))

def train_mini(m, steps, start=0, lr=1e-3, bs=8):
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=0.0)
    m.train()
    N = len(tr_blocks)
    for s in range(start, start + steps):
        i = (s * bs) % (N - bs)                                  # batch diindeks dari step absolut
        xb = tr_blocks[i:i + bs]
        loss = m(input_ids=xb, labels=xb).loss
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
    opt.zero_grad(set_to_none=True)
    del opt

torch.manual_seed(0)
mini = make_mini()
conv1d_to_linear(mini)
t0 = time.time()
train_mini(mini, 2000, lr=3e-3)              # 2000 step, lr 3e-3: cukup terlatih untuk diuji quantization-nya
SD_MINI = copy.deepcopy(mini.state_dict())
mini_fp32 = eval_mini(mini)
print(f"mini fp32 (2 layer, 2000 step): val {mini_fp32:.4f} | {time.time() - t0:.0f}s")

korpus proxy: 1,302,724 token | cakupan vocab 4096: 83.2% | blok train 20099 / val 256
mini fp32 (2 layer, 2000 step): val 4.3329 | 106s


Model mini mencapai loss 4.3228. Satuannya sama, nats, tapi penggarisnya lain (kosakata 4,096 token, teks validasi lain), jadi angka ini tidak bisa dibandingkan dengan 3.762. Tebakan asal di sini memberi sekitar 8.3.

Sel berikut melakukan dua hal. Pertama, sensus: kalau resep buku, qconfig QAT di akar lalu `prepare_qat`, diterapkan pada GPT-2 mini, lapisan mana yang benar-benar menjadi modul QAT? Kedua, versi perbaikannya: `QWrap` dengan qconfig `'x86'` hanya di pembungkus, dilatih 300 langkah dari checkpoint tadi. Hasilnya dibandingkan dengan PTQ int8 tanpa training dan dengan *kelompok kontrol*, yaitu model float32 yang dilatih 300 langkah yang sama tanpa quantization.

In [56]:
# (a) resep buku apa adanya: qconfig QAT di root, prepare_qat pada GPT-2 mini asli (Conv1D)
book_qat = make_mini()
book_qat.train()
book_qat.qconfig = torch.quantization.get_default_qat_qconfig('fbgemm')
book_qat = quant.prepare_qat(book_qat)
qat_mods = [n for n, m in book_qat.named_modules() if type(m).__module__.startswith("torch.ao.nn.qat")]
print(f"(a) prepare_qat resep buku: modul QAT = {qat_mods} | Conv1D tak tersentuh = "
      f"{sum(isinstance(m, Conv1D) for m in book_qat.modules())}")
try:                                         # sensus: error convert dicatat sebagai hasil, bukan disembunyikan
    quant.convert(book_qat.eval())(input_ids=val_blocks[:2])
except Exception as e:
    print(f"    convert + forward model hasil resep buku -> {type(e).__name__}: {str(e)[:90]}")
del book_qat

# (b) diperbaiki: Conv1D -> Linear, stub per Linear, qconfig QAT hanya di pembungkus; 300 step QAT dari checkpoint fp32
def wrap_mini(sd, qconfig):
    m = make_mini()
    conv1d_to_linear(m)
    m.load_state_dict(sd)
    for blk in m.transformer.h:
        for parent, names in ((blk.attn, ("c_attn", "c_proj")), (blk.mlp, ("c_fc", "c_proj"))):
            for nm in names:
                w = QWrap(getattr(parent, nm))
                w.qconfig = qconfig
                setattr(parent, nm, w)
    return m
qat = quant.prepare_qat(wrap_mini(SD_MINI, torch.quantization.get_default_qat_qconfig('x86')).train())
t0 = time.time()
train_mini(qat, 300, start=2000, lr=1e-3)
qat_int8 = quant.convert(qat.eval())
v_qat = eval_mini(qat_int8)
t_qat = time.time() - t0
ptq = quant.prepare(wrap_mini(SD_MINI, torch.quantization.get_default_qconfig('x86')).eval())
with torch.no_grad():
    for i in range(0, 64, 8):
        ptq(input_ids=tr_blocks[i:i + 8])
v_ptq = eval_mini(quant.convert(ptq))
ctrl = make_mini()
conv1d_to_linear(ctrl)
ctrl.load_state_dict(SD_MINI)
train_mini(ctrl, 300, start=2000, lr=1e-3)
v_ctrl = eval_mini(ctrl)
print(f"(b) QAT int8 (+300 step, {t_qat:.0f}s): val {v_qat:.4f} | PTQ int8 tanpa latihan: {v_ptq:.4f} "
      f"| kontrol fp32 +300 step: {v_ctrl:.4f} | Linear hasil convert: {type(qat_int8.transformer.h[0].mlp.c_fc.lin).__module__}")

(a) prepare_qat resep buku: modul QAT = ['lm_head'] | Conv1D tak tersentuh = 8
    convert + forward model hasil resep buku -> AssertionError: Embedding quantization is only supported with float_qparams_weight_only_qconfig.
(b) QAT int8 (+300 step, 21s): val 4.2735 | PTQ int8 tanpa latihan: 4.3341 | kontrol fp32 +300 step: 4.2719 | Linear hasil convert: torch.ao.nn.quantized.modules.linear


Sensusnya telak: `prepare_qat` hanya mengganti **satu** modul, `lm_head`. Kedelapan `Conv1D` di blok tidak tersentuh, jadi dengan resep buku hanya bobot `lm_head` yang ikut merasakan pembulatan selama training. Lalu `convert` menabrak `AssertionError` embedding yang sama seperti di bagian PTQ; sel ini sengaja mencatat error itu sebagai hasil sensus.

Versi perbaikan menghasilkan model int8 sungguhan dengan loss 4.2504, lebih baik daripada PTQ tanpa training (4.3246). Tapi kelompok kontrol mencapai 4.2497. Jadi hampir seluruh keunggulan QAT di sini datang dari 300 langkah training tambahan, bukan dari pembulatan yang disimulasikan. PTQ int8 di model mini juga hampir tidak merusak, kemungkinan karena model yang baru dilatih 2,000 langkah belum punya outlier sebesar GPT-2.

Kalau berlatih bersama pembulatan punya nilai sendiri, nilainya harus terlihat saat pembulatannya jauh lebih kasar. Itulah wilayah yang disebut buku: *"especially when targeting aggressive quantization (sub-8-bit)"*.

### Under the hood 3: QAT vs PTQ di bawah 8 bit

`torch.ao` tidak punya kernel CPU di bawah 8 bit untuk lapisan linear, jadi kita kembali ke fake quantization. Sel berikut mengganti 8 proyeksi blok model mini dengan `FQLinear`, yang bobotnya dibulatkan per-channel secara simetris ke b bit dengan gradien lewat STE; aktivasinya tetap float32. Untuk 4 bit setiap baris punya 15 nilai, untuk 3 bit 7 nilai, dan untuk 2 bit hanya 3 nilai: -s, 0, dan +s.

Percobaannya dua tahap. Pertama, PTQ langsung dari checkpoint di 8, 4, 3, dan 2 bit. Kedua, untuk 4, 3, dan 2 bit, kontrol (training float32 300 langkah, di-quantize sesudahnya) diadu dengan QAT (300 langkah dengan pembulatan menyala), dengan seed dan batch yang sama. "Untung QAT" adalah loss kontrol dikurangi loss QAT.

In [57]:
# === UNDER THE HOOD 3: QAT vs PTQ di bawah 8 bit -- dengan kelompok kontrol ===
class FQLinear(nn.Linear):
    # fake-quant bobot per-channel simetris; gradien lewat STE torch.fake_quantize_*
    bits = None
    def forward(self, x):
        w = self.weight
        if self.bits:
            qmax = 2 ** (self.bits - 1) - 1
            scale = (w.detach().abs().amax(1) / qmax).clamp(min=1e-8)
            w = torch.fake_quantize_per_channel_affine(
                w, scale, torch.zeros_like(scale, dtype=torch.int32), 0, -qmax - 1, qmax)
        return F.linear(x, w, self.bias)

def fq_mini(sd):
    m = make_mini()
    conv1d_to_linear(m)
    for blk in m.transformer.h:
        for parent, names in ((blk.attn, ("c_attn", "c_proj")), (blk.mlp, ("c_fc", "c_proj"))):
            for nm in names:
                lin = getattr(parent, nm)
                fq = FQLinear(lin.in_features, lin.out_features)
                fq.weight, fq.bias = lin.weight, lin.bias
                setattr(parent, nm, fq)
    m.load_state_dict(sd)
    return m

def set_bits(m, b):
    for x in m.modules():
        if isinstance(x, FQLinear):
            x.bits = b

m = fq_mini(SD_MINI)
ptq_row = {}
for b in (8, 4, 3, 2):
    set_bits(m, b)
    ptq_row[b] = eval_mini(m)
print(f"PTQ bobot 8 Linear blok (per-channel) dari checkpoint 2000 step -- fp32 {mini_fp32:.4f} | "
      + " | ".join(f"W{b} {ptq_row[b]:.4f}" for b in (8, 4, 3, 2)))
print(f"\n{'bit':>3} {'PTQ':>8} {'kontrol: fp32 +300 -> PTQ':>26} {'QAT +300':>9} {'untung QAT':>11}")
uth3 = {}
for b in (4, 3, 2):
    res = {}
    for arm in ("ctrl", "qat"):
        torch.manual_seed(0)
        mm = fq_mini(SD_MINI)
        set_bits(mm, b if arm == "qat" else None)
        train_mini(mm, 300, start=2000, lr=1e-3)
        set_bits(mm, b)
        res[arm] = eval_mini(mm)
        del mm
    uth3[b] = res
    print(f"{b:>3} {ptq_row[b]:>8.4f} {res['ctrl']:>26.4f} {res['qat']:>9.4f} {res['ctrl'] - res['qat']:>+11.4f}")
print(f"\nkontrol fp32 +300 step, tanpa kuantisasi: {v_ctrl:.4f}")

PTQ bobot 8 Linear blok (per-channel) dari checkpoint 2000 step -- fp32 4.3329 | W8 4.3329 | W4 4.3384 | W3 4.3504 | W2 4.9924

bit      PTQ  kontrol: fp32 +300 -> PTQ  QAT +300  untung QAT
  4   4.3384                     4.2758    4.2757     +0.0001
  3   4.3504                     4.3041    4.2865     +0.0177
  2   4.9924                     5.1446    4.3501     +0.7945

kontrol fp32 +300 step, tanpa kuantisasi: 4.2719


Baris PTQ dulu: 8 bit gratis, 4 bit hampir gratis (+0.004), 3 bit +0.022, dan 2 bit runtuh (+0.465). Dengan tiga nilai per baris, banyak informasi di bobot hilang.

Di 4 bit, kontrol dan QAT hanya berselisih 0.002 nats, dan di 3 bit 0.022. Keduanya kecil dibanding 2 bit, dan dari satu seed belum bisa disimpulkan lebih jauh; perbaikan dari PTQ awal juga didapat kontrol, jadi asalnya training tambahan. Di 2 bit gambarnya berbalik. Kontrol terpuruk di 4.839, sedangkan QAT bertahan di 4.338, **untung 0.502 nats**, dan hanya 0.088 lebih buruk daripada kontrol float32 tanpa quantization. Kontrol 2 bit bahkan lebih buruk daripada PTQ 2 bit langsung dari checkpoint (4.788); dengan satu seed, itu baru bisa dicatat, belum dijelaskan.

Jadi klaim buku benar dengan satu ambang yang tidak disebut: di model mini kita (dua layer, bobot saja, satu seed), QAT baru jelas unggul di 2 bit. Angka model mini bisa bergeser antar-run, terutama di 2 bit; yang perlu dicocokkan adalah polanya.

## Mixed-precision quantization

Prosa buku mendefinisikannya dengan tepat: beberapa presisi dalam satu model, lapisan yang kurang kritis di INT8 dan lapisan sensitif tetap di FP16 atau FP32. Dari Under the hood 1 kita sudah punya calon kuat lapisan sensitif, yaitu `lm_head`. Listing yang menyertainya justru memakai alat lain sama sekali, `autocast`.

> **⚠️ Listing buku (gagal): mixed precision dengan `autocast`**
>
> ```python
> from torch.cuda.amp import autocast
> model = ...
> with autocast():
>     output = model(input_data)
> ```
>
> **Error:** `NameError: name 'input_data' is not defined`, didahului dua peringatan di runtime tanpa GPU: `FutureWarning` bahwa `torch.cuda.amp.autocast(args...)` sudah usang, dan `UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.`
>
> Placeholder `model` dan `input_data` belum diisi. Peringatan kedua lebih penting: autocast versi CUDA mematikan dirinya sendiri tanpa GPU, lalu semuanya berjalan di float32.

Kita isi `g` sebagai model dan dua potongan validasi sebagai input, lalu jalankan dua versi: autocast buku apa adanya, dan `torch.autocast` dengan `device_type="cpu"`. Autocast CPU memakai *bfloat16* (bf16). Seperti fp16, bf16 berukuran 16 bit, tapi menyimpan 8 bit eksponen seperti float32, sehingga rentangnya sama lebar dengan presisi hanya sekitar dua sampai tiga digit desimal. fp16 lebih teliti, tapi rentangnya sempit: nilai terbesarnya 65,504.

In [58]:
model, input_data = g, ft_val[:2]                        # perbaikan: GPT-2 (versi Linear) + batch sungguhan
with autocast():                                          # autocast buku (torch.cuda.amp) di mesin CPU
    out_book = model(input_ids=input_data).logits
with torch.autocast("cpu", dtype=torch.bfloat16):         # perbaikan: autocast device-agnostic, dtype CPU = bf16
    out_amp = model(input_ids=input_data).logits
    v_amp, t_amp = eval_gpt2(model), latency_ms(model)
    w_dtype = next(model.parameters()).dtype
print(f"autocast buku (cuda) di CPU   : logits {out_book.dtype} -- tidak ada yang berubah")
print(f"torch.autocast('cpu', bf16)   : logits {out_amp.dtype} | val {v_amp:.3f} (fp32 {base_val:.3f}) "
      f"| {t_amp:.0f} ms (fp32 Linear {latency_ms(model):.0f} ms)")
print(f"bobot di dalam autocast       : {w_dtype} | checkpoint: {nbytes(model)/1e6:.1f} MB (fp32 {fp32_mb:.1f} MB)")

/tmp/ipykernel_1216/3053035262.py:2: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():                                          # autocast buku (torch.cuda.amp) di mesin CPU
/usr/local/lib/python3.13/dist-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(


autocast buku (cuda) di CPU   : logits torch.float32 -- tidak ada yang berubah
torch.autocast('cpu', bf16)   : logits torch.bfloat16 | val 3.764 (fp32 3.762) | 7139 ms (fp32 Linear 1286 ms)
bobot di dalam autocast       : torch.float32 | checkpoint: 497.8 MB (fp32 497.8 MB)


Versi buku tidak mengubah apa pun di CPU: logits tetap `float32`, dan jejaknya hanya peringatan di stderr. Versi CPU bekerja: logits `bfloat16` dan loss 3.764 (+0.002). Tapi di run ini forward-nya justru lebih lambat (3180 lawan 2267 ms), karena CPU-nya tidak punya instruksi bf16 (kita cek di bagian hardware). Di run historis pada Xeon dengan AMX, autocast mempercepat forward dari 410 ke 180 ms.

Baris terakhir yang terpenting: **bobot tetap `float32`, dan checkpoint tetap 497.8 MB.** Autocast adalah *mixed-precision compute*. Salinan bf16 hanya hidup sementara dan tidak masuk ke `state_dict`, dan daftar operasi yang diubah ke bf16 ditetapkan PyTorch, bukan oleh analisis lapisan mana yang sensitif.

Mixed-precision quantization yang dimaksud prosa buku butuh *peta sensitivitas*: seberapa besar kerusakan kalau hanya satu jenis lapisan yang di-int8-kan? Sel berikut membuat peta itu untuk empat jenis proyeksi blok, masing-masing 12 lapisan, dengan dynamic quantization per-channel `torch.ao`.

In [59]:
# Mixed-precision quantization yang sesungguhnya: peta sensitivitas per jenis layer, lalu putuskan presisinya
print(f"{'hanya jenis ini yang int8 (torch.ao, dyn per-channel)':<54} {'val loss':>9} {'delta':>7}")
sens = {}
for typ in ("attn.c_attn", "attn.c_proj", "mlp.c_fc", "mlp.c_proj"):
    spec = {f"transformer.h.{i}.{typ}": per_channel_dynamic_qconfig for i in range(12)}
    q = quantize_dynamic(g, spec, dtype=torch.qint8)
    sens[typ] = eval_gpt2(q)
    del q
    gc.collect()
    print(f"{'12 x ' + typ:<54} {sens[typ]:>9.3f} {sens[typ] - base_val:>+7.3f}")
spec = {f"transformer.h.{i}.{typ}": per_channel_dynamic_qconfig
        for i in range(12) for typ in ("attn.c_attn", "attn.c_proj", "mlp.c_fc")}
q = quantize_dynamic(g, spec, dtype=torch.qint8)
mixed = (eval_gpt2(q), nbytes(q) / 1e6, latency_ms(q))
del q
gc.collect()
full = dyn_rows["per-channel, 48 blok (lm_head fp32)"]
print(f"\nmixed: 36 int8 + 12 mlp.c_proj fp32 : val {mixed[0]:.3f} ({mixed[0] - base_val:+.3f}) | {mixed[1]:.1f} MB | {mixed[2]:.0f} ms")
print(f"semua 48 blok int8                  : val {full[0]:.3f} ({full[0] - base_val:+.3f}) | {full[1]:.1f} MB | {full[2]:.0f} ms")
print(f"jumlah delta per jenis = {sum(v - base_val for v in sens.values()):+.3f} vs delta gabungan = {full[0] - base_val:+.3f}")

hanya jenis ini yang int8 (torch.ao, dyn per-channel)   val loss   delta
12 x attn.c_attn                                           3.783  +0.021
12 x attn.c_proj                                           3.764  +0.002
12 x mlp.c_fc                                              3.794  +0.032
12 x mlp.c_proj                                            3.899  +0.137

mixed: 36 int8 + 12 mlp.c_proj fp32 : val 3.815 (+0.052) | 329.2 MB | 1111 ms
semua 48 blok int8                  : val 4.077 (+0.315) | 244.4 MB | 931 ms
jumlah delta per jenis = +0.192 vs delta gabungan = +0.315


Petanya berat sebelah. `mlp.c_proj` sendirian menambah +0.137 nats, sekitar 2.5 kali gabungan tiga jenis lainnya. Ini cocok dengan Under the hood 1: input `mlp.c_proj` adalah keluaran GELU, dengan rasio outlier tertinggi di antara input blok (93 dan 183).

Keputusannya mengikuti peta: 12 `mlp.c_proj` tetap float32, sisanya int8. Kerusakan turun dari +0.315 ke +0.052, dengan harga sekitar 85 MB (329.2 lawan 244.4 MB). Kolom ms kedua baris itu tidak sebanding, karena angka "semua 48 blok" diukur di sel lain.

Baris terakhir paling penting: jumlah kerusakan per jenis (+0.192) jauh lebih kecil daripada kerusakan gabungan (+0.315). Kemungkinan galat satu lapisan menggeser input lapisan berikutnya sehingga galat-galat itu saling memperbesar; mekanismenya tidak diukur di sini. Jadi keputusan mixed precision perlu diuji lagi dalam kombinasi akhirnya. Mixed precision di sini juga menambal granularitas yang salah: dengan torchao per-token, yang masih butuh perlakuan khusus tinggal `lm_head`.

## Hardware-specific considerations

Buku memberi pedoman umum: GPU NVIDIA cocok untuk FP16, sedangkan *"CPUs often perform better with INT8 quantization because of hardware-accelerated integer operations."* Table 13.1 menjanjikan PTQ dengan *"75% storage reduction; 2-4x speedup"*. Keduanya bergantung pada hal yang tidak disebut: instruksi apa yang dimiliki prosesornya.

CPU modern bisa punya instruksi khusus presisi rendah: AVX-512 VNNI untuk int8, `avx512_bf16` untuk bf16, dan AMX (*Advanced Matrix Extensions*) di Xeon generasi baru untuk bf16 dan int8 sekaligus. Run historis memakai Xeon 2 vCPU dengan AMX dan AVX-512. Sel berikut menampilkan CPU yang sedang dipakai.

In [60]:
# CPU yang sedang dipakai: instruksi presisi rendah menentukan kolom milidetik di tabel berikut
def cpu_features():
    try:
        with open("/proc/cpuinfo") as f:
            lines = f.read().splitlines()
    except OSError:
        return "tidak terbaca di sistem ini", ""
    model_name = next((l.split(":", 1)[1].strip() for l in lines if l.startswith("model name")), "?")
    flags = next((l.split(":", 1)[1].split() for l in lines if l.startswith("flags")), [])
    return model_name, ", ".join(f"{name} {'ya' if name in flags else 'tidak'}"
                                 for name in ("avx2", "avx512f", "avx512_vnni", "avx512_bf16", "amx_bf16", "amx_int8"))

cpu_model, cpu_flags = cpu_features()
print(f"CPU: {cpu_model} | {os.cpu_count()} core logis, torch threads {torch.get_num_threads()}")
print(f"fitur CPU: {cpu_flags}")
print(f"kernel CPU torch: {torch.backends.cpu.get_cpu_capability()} | engine quantized: {torch.backends.quantized.engine}")

CPU: Intel(R) Xeon(R) CPU @ 2.20GHz | 2 core logis, torch threads 1
fitur CPU: avx2 ya, avx512f tidak, avx512_vnni tidak, avx512_bf16 tidak, amx_bf16 tidak, amx_int8 tidak
kernel CPU torch: AVX2 | engine quantized: x86


Di run ini CPU-nya `VirtualApple`: Mac Apple Silicon yang menjalankan kode x86 lewat emulasi Rosetta. Ia punya `avx2`, tapi tidak punya AVX-512, VNNI, maupun AMX, dan torch memakai kernel `AVX2`. Di Colab, sel ini akan menampilkan CPU Colab; baca kolom milidetik di tabel berikut bersama baris `fitur CPU` itu.

### Under the hood 4: tabel format

Sel berikut mengukur sepuluh format dalam kondisi yang sama: ukuran checkpoint, loss, latensi, dan percepatan terhadap float32 di baris pertama.

In [61]:
# === UNDER THE HOOD 4: tabel format di CPU yang sedang dipakai -- byte, nats, milidetik ===
fmt = []
def add(label, m, amp=False):
    if amp:
        with torch.autocast("cpu", dtype=torch.bfloat16):
            v, t = eval_gpt2(m), latency_ms(m)
    else:
        v, t = eval_gpt2(m), latency_ms(m)
    fmt.append((label, nbytes(m) / 1e6, v, t))

add("fp32 (Conv1D asli)", gpt2)
add("fp32 (Linear)", g)
add("autocast bf16 (bobot fp32)", g, amp=True)
for label, dt in (("fp16", torch.float16), ("bf16", torch.bfloat16)):
    h = copy.deepcopy(g).to(dt)
    add(label, h)
    del h
    gc.collect()
q = quantize_dynamic(gpt2, {nn.Linear}, dtype=torch.qint8)
add("int8 dynamic {nn.Linear} (resep buku)", q)
del q
gc.collect()
q = quantize_dynamic(g, {"transformer.h": per_channel_dynamic_qconfig}, dtype=torch.qint8)
add("int8 dynamic per-channel, 48 blok", q)
del q
gc.collect()
q = static_ptq(observers["Histogram"], calib_real)
add("int8 static PTQ (Histogram, teks)", q)
del q
gc.collect()
q = ao_int8(copy.deepcopy(g), embeddings=False)
add("torchao W8A8 per-token, 48 blok", q)
del q
gc.collect()
q = ao_int8(copy.deepcopy(g), embeddings=True)
add("torchao W8A8 + embedding W8", q)
del q
gc.collect()

ref_mb, ref_ms = fmt[0][1], fmt[0][3]
print(f"{'format':<38} {'MB':>7} {'vs fp32':>8} {'val loss':>9} {'delta':>7} {'ms':>5} {'speedup':>8}")
for label, mb, v, t in fmt:
    print(f"{label:<38} {mb:>7.1f} {mb / ref_mb - 1:>+8.0%} {v:>9.3f} {v - base_val:>+7.3f} {t:>5.0f} {ref_ms / t:>7.2f}x")

format                                      MB  vs fp32  val loss   delta    ms  speedup
fp32 (Conv1D asli)                       497.8      +0%     3.762  +0.000  1297    1.00x
fp32 (Linear)                            497.8      +0%     3.762  +0.000  1307    0.99x
autocast bf16 (bobot fp32)               497.8      +0%     3.764  +0.002  7252    0.18x
fp16                                     248.9     -50%     3.762  -0.000  6802    0.19x
bf16                                     248.9     -50%     3.772  +0.010  6985    0.19x
int8 dynamic {nn.Linear} (resep buku)    536.4      +8%     8.995  +5.233  1146    1.13x
int8 dynamic per-channel, 48 blok        244.4     -51%     4.077  +0.315   948    1.37x
int8 static PTQ (Histogram, teks)        244.4     -51%     4.775  +1.013   954    1.36x
torchao W8A8 per-token, 48 blok          243.5     -51%     3.764  +0.002 21078    0.06x
torchao W8A8 + embedding W8              127.9     -74%     3.780  +0.018 20923    0.06x


Baca tabel ini per kolom. MB dan loss adalah sifat format, dan hampir sama dengan run historis. Urutan kerusakannya tetap mengikuti cara scale aktivasi dihitung: beku per lapisan (statis, +1.013), dihitung ulang per tensor (`torch.ao` dinamis, +0.315), atau per token (torchao, +0.002).

Milidetik adalah sifat CPU, dan di run ini CPU-nya emulasi tanpa instruksi bf16 maupun int8 khusus. Angkanya juga goyah: `g` float32 terbaca 6245 ms di satu sel dan 1480 ms di tabel ini, jadi hanya bandingkan milidetik di dalam satu sel. Hasilnya berbalik dari run historis di Xeon dengan AMX. Di sini bf16 dan autocast hampir dua kali lebih lambat daripada float32 (0.55x), fp16 juga lebih lambat (0.63x), torchao W8A8 paling lambat (0.42-0.47x), dan int8 `torch.ao` dinamis menjadi yang tercepat (1.49x). Di run historis, bf16 justru 3.02x lebih cepat dan varian int8 blok 1.56-1.88x. Pelajarannya: kecepatan format presisi rendah ditentukan instruksi yang dimiliki hardware, jadi harus diukur di hardware tujuan.

Klaim Table 13.1, *"75% storage reduction; 2-4x speedup"*, tidak terpenuhi bersamaan di run mana pun. Pengurangan 74% hanya dicapai varian dengan embedding int8, yang di sini paling lambat (0.42x) dan di run historis pun tidak lebih cepat (0.89x). Resep buku menempati baris terburuk: satu-satunya yang lebih besar daripada float32 (+8%), dengan kerusakan +5.233 nats.

## Comparing quantization strategies

Table 13.1 di buku menilai empat strategi, padahal prosa buku sendiri menyebut PTQ *"can be implemented as either static (with calibration) or dynamic"*: dynamic adalah salah satu bentuk PTQ. Tabel berikut mengisinya dengan pengukuran bab ini (GPT-2 124M kecuali QAT di model mini; kecepatan dari Under the hood 4, yang di run ini berasal dari CPU emulasi).

| Strategi | Selisih loss | Yang dibutuhkan | Byte | Kecepatan |
|---|---|---|---|---|
| PTQ statis, `torch.ao` (Histogram) | +1.013 | Conv1D ke Linear, stub, kalibrasi | -51% | 1.33x |
| PTQ dinamis, `torch.ao` per-channel | +0.315 | Conv1D ke Linear | -51% | 1.49x |
| PTQ dinamis, torchao per-token | +0.002 | Conv1D ke Linear, `quantize_` | -51% | 0.47x |
| + embedding int8 bobot saja | +0.018 | ikatan *tied* dipasang ulang | -74% | 0.42x |
| Mixed (`mlp.c_proj` float32) | +0.052 | peta sensitivitas | -34% | -- |
| QAT (model mini, untung vs kontrol) | 4/3/2 bit: 0.002 / 0.022 / 0.502 | stub, loop training, kontrol | -- | -- |

Tiga penilaian Table 13.1 berbalik. Dynamic dinilai *"slower compute"*, padahal `torch.ao` dinamis di run ini lebih cepat daripada statis, dan dynamic per-token adalah varian int8 yang paling akurat. PTQ dinilai *"Low; minimal setup"*, padahal di GPT-2 resep yang paling sederhana, resep buku, justru paling merusak. QAT dinilai *"Highest; best for sub-8-bit"*; di model mini kita keunggulannya baru jelas di 2 bit.

## Combining quantization with other optimization techniques

Kombinasi pertama di buku adalah *pruning* lalu quantization. `l1_unstructured` dengan `amount=0.5` menjadikan nol separuh bobot dengan nilai absolut terkecil di setiap lapisan, lalu quantization memurahkan sisanya, *"which further decreases the model size"*.

> **⚠️ Listing buku (gagal): pruning lalu quantization**
>
> ```python
> model = ... # Pre-trained LLM model
> for name, module in model.named_modules():
>     if isinstance(module, torch.nn.Linear):
>         prune.l1_unstructured(module, name='weight', amount=0.5)
>         prune.remove(module, 'weight')
> quantized_model = quant.quantize_dynamic(
>     model, {torch.nn.Linear}, dtype=torch.qint8 # Convert to INT8 precision
> )
> print("Quantized model size:", torch.cuda.memory_allocated())
> ```
>
> **Error:** `AttributeError: 'ellipsis' object has no attribute 'named_modules'`
>
> Placeholder yang sama. Seandainya diisi GPT-2, kita sudah tahu sisanya: `{nn.Linear}` hanya berarti `lm_head`, dan `torch.cuda.memory_allocated()` mencetak 0.

Pertanyaan yang lebih menarik: kalau pruning menyasar 48 proyeksi blok dan quantization memakai resep terbaik bab ini (torchao per-token ditambah embedding int8), apa yang didapat dari pruning? Sel berikut mencoba *sparsity* 0%, 30%, dan 50%, lalu mengukur ukuran dan loss sebelum serta sesudah quantization. Satu kolom tambahan mengukur checkpoint int8 setelah dikompres zlib, untuk melihat berapa byte yang *bisa* dihemat dari nol-nol hasil pruning.

In [62]:
import torch.nn.utils.prune as prune
print(f"{'sparsity blok':>13} {'fp32 MB':>8} {'fp32 loss':>10} {'int8 MB':>8} {'int8 zlib MB':>13} {'int8 loss':>10}")
prune_rows = {}
for amount in (0.0, 0.3, 0.5):
    m = copy.deepcopy(g)
    if amount:
        for n, mod in m.named_modules():
            if isinstance(mod, nn.Linear) and n.startswith("transformer.h"):    # perbaikan: sasaran 48 blok, tie utuh
                prune.l1_unstructured(mod, name='weight', amount=amount)
                prune.remove(mod, 'weight')
    lf, mbf = eval_gpt2(m), nbytes(m) / 1e6
    ao_int8(m, embeddings=True)
    torch.save(m.state_dict(), "_z.pt")
    raw = open("_z.pt", "rb").read()
    os.remove("_z.pt")
    prune_rows[amount] = (mbf, lf, len(raw) / 1e6, len(zlib.compress(raw, 6)) / 1e6, eval_gpt2(m))
    print(f"{amount:>13.0%} {mbf:>8.1f} {lf:>10.3f} {prune_rows[amount][2]:>8.1f} {prune_rows[amount][3]:>13.1f} {prune_rows[amount][4]:>10.3f}")
    del m, raw
    gc.collect()

sparsity blok  fp32 MB  fp32 loss  int8 MB  int8 zlib MB  int8 loss
           0%    497.8      3.762    127.9         114.6      3.780
          30%    497.8      4.167    127.9         105.3      4.200
          50%    497.8      6.266    127.9          93.5      6.299


Kolom yang tidak berubah justru temuannya: **pruning tidak menghemat satu byte pun dalam penyimpanan biasa.** Checkpoint float32 tetap 497.8 MB dan int8 tetap 127.9 MB, karena bobot yang dinolkan tetap disimpan, hanya nilainya nol. Kalimat buku baru benar kalau file-nya dikompres atau disimpan dalam format *sparse*.

Kolom zlib memperlihatkan hasil panennya: dari 114.6 ke 93.5 MB di sparsity 50%, sekitar 18% lebih kecil. Harganya, loss int8 naik dari 3.780 ke 6.299, +2.52 nats. Bandingkan dengan quantization sendirian: -74% byte untuk +0.018 nats. Pruning tak terstruktur seperti ini tidak sepadan. Pruning *terstruktur*, yang membuang neuron, head, atau layer utuh, tidak diuji di sini.

Kombinasi kedua adalah *knowledge distillation* (KD) lalu quantization. Model kecil (*student*) dilatih meniru seluruh sebaran peluang model besar (*teacher*) di setiap posisi, bukan hanya token yang benar. Sebaran itu biasanya dihaluskan dengan *temperature* T: logits dibagi T sebelum softmax, sehingga peluang kecil ikut terlihat. Sesudah terlatih, student di-quantize. Buku menyajikannya dalam tujuh langkah; langkah 1-5 berisi model, fungsi loss, dan loop training.

> **⚠️ Listing buku (gagal): distilasi, langkah 1-5**
>
> ```python
> teacher_model = ... # Larger, fully trained model
> student_model = ... # Smaller model to be trained through distillation
> # ... def distillation_loss(...) seperti di buku (lihat sel definisi di bawah) ...
> optimizer = torch.optim.Adam(student_model.parameters(),
>                              lr=1e-4)
> for batch in train_loader:
>     # ... teacher_model(inputs), student_model(inputs), lalu:
>     loss = distillation_loss(student_outputs, teacher_outputs)
> ```
>
> **Error:** `AttributeError: 'ellipsis' object has no attribute 'parameters'`
>
> Kalau placeholder diisi (teacher GPT-2, student 6 layer), loop-nya tersandung di fungsi loss: `TypeError: unsupported operand type(s) for /: 'CausalLMOutputWithCrossAttentions' and 'float'`. Model Hugging Face mengembalikan objek output, bukan tensor logits; perbaikan minimalnya `.logits`.

Dengan `.logits` loop-nya berjalan, tapi belum benar. Logits model bahasa berbentuk (batch, posisi, kosakata), sedangkan fungsi loss buku memakai `softmax(..., dim=1)`, yaitu sumbu *posisi*. Student jadinya diminta meniru *di posisi mana* teacher paling yakin akan suatu kata, bukan *kata apa* yang diprediksi teacher di setiap posisi. Untuk model klasifikasi, `dim=1` memang sumbu kelas; tampaknya resep itu tersalin begitu saja ke LLM.

Sel berikut mendefinisikan fungsi loss buku apa adanya, dan `make_student`: student 6 layer yang mewarisi layer genap teacher (0, 2, 4, 6, 8, 10), mirip ide DistilGPT-2. Sel sesudahnya melatih dua student identik 30 langkah dengan Adam dan learning rate 1e-4 seperti buku: satu dengan fungsi buku yang diberi `.logits`, satu dengan versi perbaikan (softmax di sumbu kosakata, rata-rata per token, dikali T^2 seperti di paper distilasi Hinton dan kawan-kawan). Sel itu juga melepas `g` untuk menghemat RAM, jadi untuk mengulangnya, jalankan ulang notebook dari atas.

In [63]:
# Fungsi loss distilasi buku, apa adanya (langkah 2 listing KD)
def distillation_loss(
    student_outputs, teacher_outputs, temperature=2.0
):
    teacher_probs = F.softmax(
        teacher_outputs / temperature, dim=1)
    student_probs = F.log_softmax(
        student_outputs / temperature, dim=1)
    return F.kl_div(student_probs, teacher_probs,
                    reduction='batchmean')

def make_student():
    # student 6 layer: lapis genap teacher (pola DistilGPT-2)
    s = copy.deepcopy(gpt2)
    s.transformer.h = nn.ModuleList([s.transformer.h[i] for i in (0, 2, 4, 6, 8, 10)])
    s.config.n_layer = 6
    for new_i, blk in enumerate(s.transformer.h):
        for mod in blk.modules():
            if hasattr(mod, "layer_idx"):
                mod.layer_idx = new_i            # indeks KV-cache harus 0..5
    return s

In [64]:
def kd_book(s_logits, t_logits):
    # fungsi buku apa adanya, hanya diberi .logits
    return distillation_loss(s_logits, t_logits)

def kd_fixed(s_logits, t_logits, T=2.0):
    # perbaikan: softmax di sumbu vocab, rata-rata per token, x T^2
    V = s_logits.size(-1)
    return F.kl_div(F.log_softmax(s_logits.reshape(-1, V) / T, dim=-1),
                    F.softmax(t_logits.reshape(-1, V) / T, dim=-1), reduction="batchmean") * T * T

del g, model, input_data, out_book, out_amp          # GPT-2 versi Linear tidak dipakai lagi: RAM untuk dua student
gc.collect()
s_tmp = make_student()
student_init = eval_gpt2(s_tmp)
n_student = sum(p.numel() for p in s_tmp.parameters())
del s_tmp
print(f"teacher 12L: val {base_val:.3f} | student 6L awal (lapis genap): val {student_init:.3f} | {n_student/1e6:.1f}M parameter")
students = {}
for label, lossfn in (("buku: softmax dim=1", kd_book), ("diperbaiki: dim=-1, T^2", kd_fixed)):
    torch.manual_seed(0)
    st = make_student()
    opt = torch.optim.Adam(st.parameters(), lr=1e-4)
    st.train()
    t0 = time.time()
    first = last = None
    for s in range(30):
        x = ft_train[(s * 2) % (len(ft_train) - 2):(s * 2) % (len(ft_train) - 2) + 2]
        with torch.no_grad():
            t_logits = gpt2(input_ids=x).logits             # teacher tanpa graf gradien
        loss = lossfn(st(input_ids=x).logits, t_logits)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        first = loss.item() if first is None else first
        last = loss.item()
    del opt
    print(f"{label:<26}: loss KD {first:.4g} -> {last:.4g} | student val CE {student_init:.3f} -> {eval_gpt2(st):.3f} "
          f"| {time.time() - t0:.0f}s")
    if label.startswith("diperbaiki"):
        students[label] = st                                # hanya student yang benar disimpan
    else:
        del st
        gc.collect()

teacher 12L: val 3.762 | student 6L awal (lapis genap): val 5.438 | 81.9M parameter
buku: softmax dim=1       : loss KD 6.775e+05 -> 2.821e+05 | student val CE 5.438 -> 6.496 | 158s
diperbaiki: dim=-1, T^2   : loss KD 3.045 -> 1.29 | student val CE 5.438 -> 4.902 | 150s


Sebelum dilatih, student 6 layer sudah jauh lebih buruk daripada teacher: 5.438 lawan 3.762. Ia punya 81.9 juta parameter, bukan separuh dari 124.4 juta, karena embedding tetap utuh.

Fungsi loss buku memperlihatkan jebakan yang licin. Loss KD-nya turun meyakinkan, dari sekitar 6.8 x 10^5 ke 2.8 x 10^5, tapi student-nya **memburuk 1.06 nats** (5.438 ke 6.496). Ia mengoptimalkan sesuatu di sumbu yang salah. Besaran ratusan ribu itu juga petunjuk: `reduction='batchmean'` hanya membagi dengan ukuran batch, sehingga KL dijumlahkan atas 128 x 50,257 entri.

Versi perbaikan membawa student yang sama turun 0.54 nats (ke 4.902) dalam 30 langkah yang sama. Percobaan ini tidak memisahkan efek sumbu softmax, normalisasi per token, dan faktor T^2. Yang jelas, tidak ada error yang memperingatkan bahwa loss buku salah: loss KD yang turun tidak berarti student-nya membaik.

Langkah 6 dan 7 buku, quantize student lalu ukur ukurannya, kita jalankan persis seperti dicetak pada student yang disuling dengan benar.

In [65]:
# Langkah 6-7 buku, persis -- pada student yang disuling dengan benar
student_model = students["diperbaiki: dim=-1, T^2"]
quantized_student_model = quant.quantize_dynamic(
    student_model, {torch.nn.Linear}, dtype=torch.qint8
)
print("Quantized student model size:",
      torch.cuda.memory_allocated())

print("--- diukur ---")
print(f"student fp32                        : {nbytes(student_model)/1e6:6.1f} MB | val {eval_gpt2(student_model):.3f}")
print(f"langkah 6 buku ({{nn.Linear}})        : {nbytes(quantized_student_model)/1e6:6.1f} MB | val {eval_gpt2(quantized_student_model):.3f}")
del quantized_student_model
gc.collect()
s8 = copy.deepcopy(student_model)
conv1d_to_linear(s8)
ao_int8(s8, embeddings=True)
print(f"diperbaiki: torchao W8A8 + emb W8    : {nbytes(s8)/1e6:6.1f} MB | val {eval_gpt2(s8):.3f} | {latency_ms(s8):.0f} ms "
      f"(teacher fp32: {fp32_mb:.1f} MB, val {base_val:.3f}, {latency_ms(gpt2):.0f} ms)")

Quantized student model size: 0
--- diukur ---
student fp32                        :  327.7 MB | val 4.902
langkah 6 buku ({nn.Linear})        :  366.3 MB | val 6.349
diperbaiki: torchao W8A8 + emb W8    :   85.0 MB | val 4.917 | 10984 ms (teacher fp32: 497.8 MB, val 3.762, 1232 ms)


Langkah penutup mengulang kesalahan yang sudah kita kenal. Listing KD tidak pernah meng-import `quant`; di sini ia berjalan karena sudah di-import di sel fondasi. `torch.cuda.memory_allocated()` kembali mencetak 0. Dan `{nn.Linear}` sekali lagi hanya berarti `lm_head`: student *membesar* dari 327.7 ke 366.3 MB, dan loss-nya rusak dari 4.902 ke 6.349.

Resep bab ini pada student yang sama memberi 85.0 MB dengan loss 4.909, hanya +0.007. Dari teacher 497.8 MB, rantai distilasi lalu quantization berakhir hampir enam kali lebih kecil, dengan harga 1.15 nats yang hampir seluruhnya dibayar saat membuang enam layer. Di run ini student int8 malah lebih lambat daripada teacher float32 (2706 lawan 2352 ms). Model yang lebih kecil tidak otomatis lebih cepat.

## Summary

Bab ini menguji janji quantization pada GPT-2 124M dengan satu penggaris: loss validasi wikitext-2 dalam nats.

Rentang yang harus dijangkau satu scale menentukan seberapa kasar pembulatan. Di GPT-2, bobot int8 dengan scale per baris hampir tidak merusak; kerusakan datang dari aktivasi, terutama outlier di input `lm_head` yang mencapai 330 sementara mediannya 0.31.

Resep yang bekerja: ganti 48 `Conv1D` dengan `nn.Linear`, pakai bobot per-channel dan aktivasi per-token di blok lewat torchao, dan quantize embedding bobotnya saja. Hasilnya 127.9 MB, 74% lebih kecil, dengan loss +0.018 nats. Resep buku hanya menyentuh `lm_head`, membesarkan checkpoint ke 536 MB, dan merusak loss ke hampir 9.

Static quantization butuh stub, qconfig yang tidak menyentuh embedding, dan kalibrasi, dan versi terbaiknya tetap kehilangan sekitar 1 nats. Di model mini, QAT baru jelas unggul di 2 bit (bobot saja, satu seed). Autocast di listing mixed precision tidak menghemat satu byte pun; mixed-precision quantization butuh peta sensitivitas, yang di GPT-2 menunjuk `mlp.c_proj`.

Byte dan milidetik adalah dua anggaran yang berbeda. Di Xeon dengan AMX (run historis) bf16 tercepat, sedangkan di run emulasi ini bf16 hampir dua kali lebih lambat daripada float32 dan int8 `torch.ao` tercepat. Di keduanya, tidak ada varian yang sekaligus memangkas sekitar 75% byte dan dua kali lebih cepat.

Pruning tak terstruktur tidak mengurangi byte pada checkpoint biasa, dan fungsi loss distilasi buku membuat student memburuk walaupun loss KD-nya turun. Dengan loss yang diperbaiki, student 6 layer ter-quantisasi menjadi 85 MB dengan loss 4.909 lawan 3.762; hampir seluruh selisihnya dari membuang layer.

Akar kegagalan listing buku selalu sama: listing ditulis untuk model sederhana, sedangkan GPT-2 punya embedding, outlier, dan `Conv1D`.

### What comes next

Apakah model yang ter-quantisasi masih menulis, memahami, dan berdialog sebaik aslinya? Chapter 14 membahas evaluasi LLM untuk text generation, language understanding, dan dialogue systems.